# 이온 밀 식각 장비 예지보전 EDA — 레시피 스텝 기준

| 항목 | 내용 |
|---|---|
| 서비스 목표 | 웨이퍼 뒷면 헬륨 냉각(flowcool) 계통 고장 3종을 **고장 전에** 알리는 것 |
| 데이터 | `data/processed/step/` — `recipe_step`이 이어지는 구간 하나를 한 행으로 요약 (train 20대, 2,437,615행, 원본의 약 3%) |
| 분석 대상 | **식각 중(셔터 1)만** — 원본에서 셔터 1 행만 먼저 거른 뒤 스텝 구간으로 요약한 `data/processed/step_s1/` 526,268행 (구간 수 21.6%, 시간 61.0%). 무작위 샘플링 없음. 근거와 빠지는 정보는 2-3 |
| 보조 데이터 | `data/processed/prefault/` (고장 24시간 전 원본 4초 행), `data/processed/faults.csv` |
| 이전 EDA | `eda/eda_v1.ipynb` (1분 요약 기준). 1분 안에서 스텝이 바뀌면 값이 섞이는 문제가 있어 스텝 기준으로 다시 본다 |
| 상태 | 초안 |

## 1. 분석 목적

| 질문 | 이 답으로 정할 것 |
|---|---|
| Q1. 레시피·스텝은 센서 값에 얼마나 영향을 주는가? | 정규화 기준(스텝별 z값)을 쓸지 |
| Q2. 어떤 센서가 고장 전과 평소를 구분하는가? | 입력 센서 후보 |
| Q3. 고장 징후는 고장 몇 시간 전부터 보이는가? 갑자기 오는가, 서서히 오는가? | 라벨 창 길이 |
| Q4. 고장 직전에 기록이 있는가? 고장 간격은 어떤가? | 평가에서 뺄 사건, 사건 정의 |
| Q5. 장비·M01/M02·셔터 상태에 따라 다른가? | 검증 분할, 사용 구간(셔터 1만 → 2-3에서 결정) |

질문별 답은 **5-0**에 정리했다.

In [1]:
# ── 1) 라이브러리 ──────────────────────────────────────────────
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import rankdata
from IPython.display import display, Markdown

# ── 2) 경로: 노트북을 프로젝트 루트나 eda/ 어디서 열어도 data/processed를 찾는다 ──
cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, cwd.parent) if (path / 'data/processed/step').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=520):
    """그림은 파일로 저장하지 않고 노트북에만 표시한다."""
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=70))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

# ── 3) 상수 ──────────────────────────────────────────────────────
# 연속형 센서 16개 (원본 컬럼 이름)
SENSORS = ['IONGAUGEPRESSURE', 'ETCHBEAMVOLTAGE', 'ETCHBEAMCURRENT', 'ETCHSUPPRESSORVOLTAGE', 'ETCHSUPPRESSORCURRENT',
           'FLOWCOOLFLOWRATE', 'FLOWCOOLPRESSURE', 'ETCHGASCHANNEL1READBACK', 'ETCHPBNGASREADBACK',
           'FIXTURETILTANGLE', 'ROTATIONSPEED', 'ACTUALROTATIONANGLE', 'ETCHSOURCEUSAGE', 'ETCHAUXSOURCETIMER',
           'ETCHAUX2SOURCETIMER', 'ACTUALSTEPDURATION']
COUNTERS = ['ETCHSOURCEUSAGE', 'ETCHAUXSOURCETIMER', 'ETCHAUX2SOURCETIMER']   # 계속 쌓이는 누적 카운터
NAMES = {'IONGAUGEPRESSURE': '챔버 압력', 'ETCHBEAMVOLTAGE': '빔 전압', 'ETCHBEAMCURRENT': '빔 전류',
         'ETCHSUPPRESSORVOLTAGE': '억제 전압', 'ETCHSUPPRESSORCURRENT': '억제 전류', 'FLOWCOOLFLOWRATE': '냉각 유량',
         'FLOWCOOLPRESSURE': '냉각 압력', 'ETCHGASCHANNEL1READBACK': '아르곤 유량(소스)', 'ETCHPBNGASREADBACK': '아르곤 유량(PBN)',
         'FIXTURETILTANGLE': '웨이퍼 기울기', 'ROTATIONSPEED': '회전 속도', 'ACTUALROTATIONANGLE': '회전 각도',
         'ETCHSOURCEUSAGE': '그리드 사용량', 'ETCHAUXSOURCETIMER': '실드 사용량1', 'ETCHAUX2SOURCETIMER': '실드 사용량2',
         'ACTUALSTEPDURATION': '단계 소요시간'}
# 고장 3종: 원래 이름 -> 짧은 이름
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
FSHORT = dict(zip(FAULTS, ['압력 저하', '압력 과다', '누설']))
ORDER = [FSHORT[fault] for fault in FAULTS]
TTF = {fault: 'TTF_' + fault for fault in FAULTS}        # 고장까지 남은 시간(초) 열 이름
FEAT = [sensor + '_mean' for sensor in SENSORS]          # 스텝 구간 안 4초 값의 평균 열
HOUR, DAY = 3600, 86400                                  # 시간 단위(초)
C_FAIL, C_OK, C_GREY = '#eb6834', '#2a78d6', '#7a7f87'
FCOLOR = dict(zip(ORDER, [C_OK, C_FAIL, C_GREY]))

def label(tool):
    """고장 파일의 장비 이름을 step 파일 형식으로 바꾼다. 01M01 -> M01_01"""
    return f'{tool[2:]}_{tool[:2]}'

# ── 4) 스텝 구간 데이터: 20대 전부 (샘플링 없음) ──────────────────
load_cols = ['Tool', 'time_start', 'time_end', 'duration_s', 'n_rows', 'recipe', 'recipe_step', 'shutter'] + list(TTF.values()) + FEAT
df = pd.concat([pd.read_parquet(path, columns=load_cols) for path in sorted((PROC / 'step').glob('*.parquet'))],
               ignore_index=True)
df['Tool'] = df.Tool.astype('category')
df['M'] = df.Tool.astype(str).str[:3]                                  # 장비 종류 M01 / M02
df = df.sort_values(['Tool', 'time_start']).reset_index(drop=True)     # 장비별 시간순 정렬 (searchsorted에 필요)

# ── 5) 고장 기록 ─────────────────────────────────────────────────
faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.map(label)
faults['fault_type'] = faults.fault_name.str.replace(' [NoWaferID]', '', regex=False)   # 같은 고장의 표기 차이 통일
# 같은 장비·같은 시각·같은 종류로 두 번 찍힌 기록은 하나로 본다
events = (faults.drop_duplicates(['Tool', 'time', 'fault_type'])
          .sort_values(['Tool', 'fault_type', 'time']).reset_index(drop=True))

# ── 6) 불러온 데이터가 가공 기록(manifest)과 맞는지 확인 ─────────────
manifest = json.loads((PROC / 'manifest.json').read_text(encoding='utf-8'))
assert len(df) == sum(tool_info['step_rows'] for tool_info in manifest.values()) and df.Tool.nunique() == 20
assert len(faults) == 1236 and len(events) == 1211
RAW_ROWS = int(df.n_rows.sum())                                        # 원본 4초 행 수 합계
df_all = df                                                           # 전체 구간 (2-3에서만 사용)

# ── 7) 여러 절에서 쓰는 함수 ──────────────────────────────────────
def nearest_fault_dist(segments, fault_events):
    """구간마다 가장 가까운 고장 기록(앞 또는 뒤)까지의 시간(초). 고장이 없는 장비는 무한대."""
    dist = np.full(len(segments), np.inf)
    seg_start, seg_end = segments.time_start.to_numpy(), segments.time_end.to_numpy()
    for tool, row_idx in segments.groupby('Tool', observed=True).indices.items():
        fault_times = np.sort(fault_events.loc[fault_events.Tool == tool, 'time'].unique())
        if len(fault_times) == 0:
            continue
        # 구간 시작 직전 고장까지의 시간
        prev_pos = np.searchsorted(fault_times, seg_start[row_idx])
        dist_prev = np.where(prev_pos > 0, seg_start[row_idx] - fault_times[np.maximum(prev_pos - 1, 0)], np.inf)
        # 구간 끝 직후 고장까지의 시간
        next_pos = np.searchsorted(fault_times, seg_end[row_idx])
        dist_next = np.where(next_pos < len(fault_times),
                             fault_times[np.minimum(next_pos, len(fault_times) - 1)] - seg_end[row_idx], np.inf)
        dist[row_idx] = np.minimum(dist_prev, dist_next)
    return dist

def auc(positive, negative):
    """순위 기반 AUC: 고장 전 값(positive)이 평소 값(negative)보다 클 확률. 0.5 = 구분 못 함. 각 20개 미만이면 NaN."""
    positive = positive[np.isfinite(positive)]
    negative = negative[np.isfinite(negative)]
    if len(positive) < 20 or len(negative) < 20:
        return np.nan
    ranks = rankdata(np.r_[positive, negative])
    return (ranks[:len(positive)].sum() - len(positive) * (len(positive) + 1) / 2) / (len(positive) * len(negative))


### 데이터 한눈에 보기

학습 장비 20대 전체를 세 가지로 나눠 본다. 고장 기록은 중복을 뺀 값이고, 셔터 상태는 원본 행(약 4초당 1행, 즉 시간) 기준이다.

In [2]:
# ① 고장 종류별 기록 수 (중복 제외)
fault_type_counts = events.fault_type.map(FSHORT).value_counts().reindex(ORDER)

# ② 장비별 고장 기록 수: 상위 3대 + 나머지 17대
tool_fault_counts = events.Tool.astype(str).value_counts()
top_tools = list(tool_fault_counts.index[:3])
tool_share = pd.concat([tool_fault_counts[top_tools],
                        pd.Series({f'나머지 {len(tool_fault_counts) - 3}대': tool_fault_counts.iloc[3:].sum()})])

# ③ 셔터 상태별 시간: 원본 행 수(n_rows)를 더한다 → 시간 기준 비율
shutter_label = df.shutter.map({1: '1 (식각 중)', 0: '0'}).fillna('2·3·255·결측')
shutter_time = df.n_rows.groupby(shutter_label).sum().reindex(['1 (식각 중)', '0', '2·3·255·결측'])

# 그림 0-1: 파이 3개를 가로로 배치
pie_specs = [('고장 종류 (기록 수)', fault_type_counts, [C_OK, C_FAIL, C_GREY]),
             ('장비별 고장 (기록 수)', tool_share, ['#2a78d6', '#eb6834', '#4c9a52', '#dfe3e8']),
             ('셔터 상태 (원본 행 = 시간)', shutter_time, [C_FAIL, C_OK, '#dfe3e8'])]
pie_x_ranges = [(0, .30), (.35, .65), (.70, 1)]
fig = go.Figure()
for (x_left, x_right), (title, counts, colors) in zip(pie_x_ranges, pie_specs):
    ratios = counts / counts.sum()
    fig.add_trace(go.Pie(labels=counts.index, values=counts.values, sort=False, direction='clockwise', hole=.35,
                         domain=dict(x=[x_left, x_right], y=[0, .74]),           # 위쪽은 제목 자리로 비운다
                         marker=dict(colors=colors, line=dict(color='white', width=1)),
                         # 5% 미만 조각은 글자가 겹치므로 생략한다(마우스를 올리면 보임)
                         text=[f'{slice_name}<br>{ratio:.1%}' if ratio >= .05 else ''
                               for slice_name, ratio in zip(counts.index, ratios)],
                         textinfo='text', textposition='inside', showlegend=False, name=title,
                         hovertemplate='%{label}<br>%{value:,} (%{percent:.1%})<extra></extra>'))
    fig.add_annotation(x=(x_left + x_right) / 2, y=.9, xref='paper', yref='paper', showarrow=False, font=dict(size=14),
                       text=f'<b>{title}</b><br>전체 {counts.sum():,}')
fig.update_layout(title='그림 0-1. 데이터 구성 (학습 장비 20대, 5% 미만 조각은 글자 생략)')
show(fig, 560)
print(f'고장 상위 3대({", ".join(top_tools)})가 고장 기록의 {tool_fault_counts[top_tools].sum() / tool_fault_counts.sum():.1%}')

고장 상위 3대(M02_03, M01_09, M01_08)가 고장 기록의 51.4%


## 2. 데이터셋 설명

### 2-1. 출처·범위

[PHM Society 2018 Data Challenge](https://www.phmsociety.org/events/conference/phm/18/data-challenge)의 이온 밀 식각 장비 데이터다. 정답이 있는 train 20대만 쓴다(test 5대는 정답 비공개).

`src/make_processed.py`가 원본(약 4초 간격)을 **`recipe_step`이 바뀔 때마다 새 행**으로 묶었다. 한 행은 같은 스텝이 이어진 구간 하나이고, 구간 안의 4초 기록을 요약했다. 일부 행을 뽑는 샘플링이 아니라 전부를 요약한 집계다.

In [3]:
# 장비 × 고장 종류별 기록 수 (중복 제외)
fault_counts_by_type = (events.assign(fault_short=events.fault_type.map(FSHORT))
                        .pivot_table(index='Tool', columns='fault_short', values='time', aggfunc='size',
                                     fill_value=0, observed=True)
                        .reindex(columns=ORDER, fill_value=0))

# 장비별 규모 표: 구분(M01/M02), 스텝 구간 수, 원본 행 수, 기록 기간, 고장 수
by_tool = df.groupby('Tool', observed=True)
tool_scale = pd.DataFrame({
    '구분': by_tool.M.first(),
    '스텝 구간': by_tool.size(),
    '원본 행': by_tool.n_rows.sum(),
    '기간(일)': ((by_tool.time_end.max() - by_tool.time_start.min()) / DAY).round(0).astype(int),
}).join(fault_counts_by_type)
tool_scale['고장 합계'] = tool_scale[ORDER].sum(axis=1)
total_row = tool_scale.drop(columns='구분').sum()
total_row['기간(일)'] = np.nan                                        # 기간은 더하면 의미가 없으므로 비운다
display(pd.concat([tool_scale, total_row.rename('합계').to_frame().T.assign(구분='')])
        .astype({'스텝 구간': int, '원본 행': int}))

# M01 / M02별 합계와 고장 비중
machine_table = tool_scale.groupby('구분')[['스텝 구간', '원본 행', '고장 합계']].sum()
machine_table['고장 비중(%)'] = (machine_table['고장 합계'] / machine_table['고장 합계'].sum() * 100).round(1)
display(machine_table)

fig = px.bar(tool_scale.reset_index(), x='Tool', y='스텝 구간', color='구분', color_discrete_sequence=[C_OK, C_FAIL],
             title='그림 2-1. 장비별 스텝 구간 수')
show(fig, 400)

,구분,스텝 구간,원본 행,기간(일),압력 저하,압력 과다,누설,고장 합계
M01_01,M01,71389,3123826,460.0,13.0,5.0,1.0,19.0
M01_02,M01,59912,2865534,338.0,9.0,7.0,1.0,17.0
M01_03,M01,85003,3432177,444.0,40.0,51.0,0.0,91.0
M01_04,M01,165109,5097823,467.0,7.0,1.0,1.0,9.0
M01_05,M01,164949,4778645,455.0,14.0,7.0,7.0,28.0
M01_06,M01,115423,3811931,445.0,10.0,3.0,3.0,16.0
M01_07,M01,191981,5300936,453.0,15.0,2.0,18.0,35.0
M01_08,M01,87811,3715869,414.0,68.0,32.0,17.0,117.0
M01_09,M01,141885,4565256,463.0,129.0,19.0,2.0,150.0
M01_10,M01,101969,3166010,465.0,13.0,1.0,16.0,30.0


,스텝 구간,원본 행,고장 합계,고장 비중(%)
구분,,,,
M01,1185431,39858007,512,42.3
M02,1252184,42325963,699,57.7


### 2-2. 컬럼 설명

원본 24열(+ 정답 TTF 3열)을 62열로 바꿨다. 연속형 센서 16개는 train 20대 전체 기준으로 **표준화된 값**이라 실제 단위는 알 수 없다.

| 구분 | 열 | 만드는 법 |
|---|---|---|
| 장비 | `Tool` | `M01_01` 형식 (M01/M02 × 번호 01~10) |
| 시각 | `time_start`, `time_end`, `duration_s`, `n_rows` | 구간의 첫·끝 시각(초), 지속시간, 원본 행 수 |
| 공정 정보 | `stage`, `Lot`, `runnum`, `recipe`, `recipe_step`, `shutter` | 구간의 첫 값. `recipe_step`은 구간 안에서 바뀌지 않는다 |
| 정답 | `TTF_…` 3개 | 구간 안 최솟값 = 구간 끝에서 고장까지 남은 시간. 결측은 "기간 안에 고장 없음" |
| 센서 | `<센서>_mean`, `_min`, `_max` (16 × 3 = 48열) | 구간 안 4초 값의 평균·최소·최대 |

| 센서 | 뜻 | 센서 | 뜻 |
|---|---|---|---|
| `FLOWCOOLPRESSURE` | 냉각(헬륨) 압력 | `FLOWCOOLFLOWRATE` | 냉각 유량 |
| `IONGAUGEPRESSURE` | 챔버 압력 | `ETCHBEAMVOLTAGE`·`CURRENT` | 빔 전압·전류 |
| `ETCHSUPPRESSORVOLTAGE`·`CURRENT` | 억제 전압·전류 | `ETCHGASCHANNEL1READBACK`·`ETCHPBNGASREADBACK` | 아르곤 유량(소스·PBN) |
| `FIXTURETILTANGLE` | 웨이퍼 기울기 | `ROTATIONSPEED`·`ACTUALROTATIONANGLE` | 회전 속도·각도 |
| `ETCHSOURCEUSAGE`, `ETCHAUXSOURCETIMER`, `ETCHAUX2SOURCETIMER` | 누적 사용량(카운터) | `ACTUALSTEPDURATION` | 단계 소요시간 |

셔터(`shutter`)는 이온 빔과 웨이퍼 사이의 가림막이다. 선행 연구는 값 1을 식각 중으로 본다(2-3).

### 2-3. 분석 대상 — 식각 중(셔터 1) 구간만 쓴다

셔터(`shutter`)는 이온 빔과 웨이퍼 사이의 가림막이다. 공식 설명은 없지만 선행 연구(TCS 2018, UMD 2025)는 값 1을 **식각 중**으로 보고 그 구간만 썼다. 이 노트북도 **3장부터는 셔터 1 구간만** 본다. 무작위로 뽑는 샘플링이 아니라 조건으로 거르는 필터링이다.

거르는 순서: **원본 4초 행에서 셔터 1만 먼저 거르고 → 스텝 구간으로 요약**한다(`data/processed/step_s1/`). 셔터 1이 끊기면 같은 스텝이어도 다른 구간이 된다. 전체를 먼저 묶고 첫 셔터 값으로 거르는 방식과 차이가 있는지 마지막에 확인한다.

이 절만 **전체 구간**으로 아래 네 가지를 본다.
1. 셔터 값별 비율 (구간 수 / 시간)
2. 식각 중과 셔터 0의 센서 값 차이 (효과 크기)
3. 냉각 압력·유량의 고장 구분력: 전체 구간 vs 셔터 1
4. 셔터 1만 쓰면 **빠지는 정보**: 고장 직전 셔터 상태, 고장 직전 기록 유무

In [4]:
shutter_code = df_all.shutter.fillna(-1).to_numpy()
all_weight = df_all.n_rows.to_numpy()                      # 가중치 = 원본 행 수 → 시간 기준
prefault_all = {fault: (df_all[TTF[fault]] <= DAY).to_numpy() for fault in FAULTS}
SH_LABEL = {-1: '결측', 0: '0', 1: '1 (식각 중)', 2: '2', 3: '3', 255: '255'}

# ── 1) 셔터 값별 비율 ────────────────────────────────────────────
def shutter_share(row_mask, weight):
    # row_mask 구간 안에서 셔터 값별 비율(%). weight가 1이면 구간 수, n_rows면 시간 기준
    weight_by_shutter = pd.Series(weight[row_mask]).groupby(shutter_code[row_mask]).sum()
    return weight_by_shutter / weight_by_shutter.sum() * 100

all_rows = np.ones(len(df_all), bool)
shutter_columns = {'전체 · 구간 수': shutter_share(all_rows, np.ones(len(df_all))), '전체 · 시간': shutter_share(all_rows, all_weight)}
for fault in FAULTS:
    shutter_columns[f'{FSHORT[fault]} 24시간 전 · 시간'] = shutter_share(prefault_all[fault], all_weight)
shutter_table = pd.DataFrame(shutter_columns).fillna(0).round(1)
shutter_table.index = [SH_LABEL.get(int(shutter_value), str(shutter_value)) for shutter_value in shutter_table.index]
shutter_table.index.name = '셔터 값 (%)'
display(shutter_table)

# ── 2) 식각 중(셔터 1) vs 셔터 0 센서 값 차이 ────────────────────────
is_etching_all, is_shutter0_all = shutter_code == 1, shutter_code == 0
shutter_diff_rows = []
for sensor, mean_col in zip(SENSORS, FEAT):
    sensor_values = df_all[mean_col].to_numpy()
    etching_values, shutter0_values = sensor_values[is_etching_all], sensor_values[is_shutter0_all]
    # 효과 크기 = (식각 중 평균 − 셔터 0 평균) / 두 집단 표준편차의 평균(제곱 평균의 제곱근)
    pooled_std = np.sqrt((np.nanvar(etching_values) + np.nanvar(shutter0_values)) / 2)
    effect_size = (np.nanmean(etching_values) - np.nanmean(shutter0_values)) / pooled_std if pooled_std > 0 else np.nan
    shutter_diff_rows.append({'센서': NAMES[sensor], '식각 중 중앙값': np.nanmedian(etching_values),
                              '셔터 0 중앙값': np.nanmedian(shutter0_values), '차이(표준편차 단위)': effect_size})
shutter_diff = (pd.DataFrame(shutter_diff_rows).set_index('센서')
                .sort_values('차이(표준편차 단위)', key=lambda col: col.abs(), ascending=False))
print(f'구간 수: 식각 중 {is_etching_all.sum():,} · 셔터 0 {is_shutter0_all.sum():,}')
display(shutter_diff.round(2))

bar_order = shutter_diff.iloc[::-1]
fig = go.Figure(go.Bar(x=bar_order['차이(표준편차 단위)'], y=bar_order.index, orientation='h',
                       marker_color=[C_FAIL if abs(value) >= 0.8 else C_GREY for value in bar_order['차이(표준편차 단위)']],
                       hovertemplate='%{y}<br>%{x:.2f} 표준편차<extra></extra>'))
for line_x in (-0.8, 0.8):
    fig.add_vline(x=line_x, line=dict(color='#999', dash='dot', width=1))
fig.add_vline(x=0, line=dict(color='#444', width=1))
fig.update_layout(title='그림 2-3a. 식각 중(셔터 1) − 셔터 0 센서 값 차이 (효과 크기, 점선 = ±0.8 큰 차이)',
                  xaxis_title='차이(표준편차 단위, 양수 = 식각 중에 높음)', yaxis_title='')
show(fig, 520)

# ── 3) 냉각 압력·유량 구분력: 전체 구간 vs 셔터 1 ─────────────────────
# z값 기준은 3-2와 같다: 장비·레시피·스텝·셔터별 평소(고장에서 24시간 넘게) 구간, 30개 이상
COMPARE_SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE']
is_normal_all = nearest_fault_dist(df_all, events) > DAY
ctx_all = df_all.assign(_rs=df_all.recipe_step.fillna(-1), _sh=df_all.shutter.fillna(-1)) \
                .groupby(['Tool', 'recipe', '_rs', '_sh'], observed=True, sort=False).ngroup().to_numpy()
normal_frame = df_all.loc[is_normal_all, [sensor + '_mean' for sensor in COMPARE_SENSORS]].assign(ctx=ctx_all[is_normal_all])
normal_group = normal_frame.groupby('ctx')
enough_normal_ctx = normal_group.size() >= 30
z_all = {}
for sensor in COMPARE_SENSORS:
    ctx_mean = normal_group[sensor + '_mean'].mean().reindex(ctx_all).to_numpy()
    ctx_std = normal_group[sensor + '_mean'].std().where(enough_normal_ctx).reindex(ctx_all).to_numpy()
    z_all[sensor] = (df_all[sensor + '_mean'].to_numpy() - ctx_mean) / np.where(ctx_std > 1e-6, ctx_std, np.nan)

compare_rng = np.random.default_rng(0)
compare_rows = []
for scope_name, scope_mask in [('전체 구간', all_rows), ('셔터 1', is_etching_all)]:
    for tool, row_idx in df_all.groupby('Tool', observed=True).indices.items():
        normal_idx = row_idx[is_normal_all[row_idx] & scope_mask[row_idx]]
        if len(normal_idx) > 50_000:                         # 평소 구간은 장비당 최대 5만 개 표본
            normal_idx = compare_rng.choice(normal_idx, 50_000, replace=False)
        for fault in FAULTS:
            prefault_idx = row_idx[prefault_all[fault][row_idx] & scope_mask[row_idx]]
            for sensor in COMPARE_SENSORS:
                compare_rows.append({'범위': scope_name, '센서': NAMES[sensor], '고장': FSHORT[fault],
                                     'auc': auc(z_all[sensor][prefault_idx], z_all[sensor][normal_idx])})
scope_auc = (pd.DataFrame(compare_rows).groupby(['센서', '고장', '범위']).auc.median().unstack('범위')
             [['전체 구간', '셔터 1']].reindex(ORDER, level='고장'))
scope_auc['|AUC−0.5| 전체'] = (scope_auc['전체 구간'] - .5).abs()
scope_auc['|AUC−0.5| 셔터 1'] = (scope_auc['셔터 1'] - .5).abs()
display(Markdown('**장비별 AUC 중앙값** (0.5 = 구분 못 함, 0.5에서 멀수록 잘 구분)'))
display(scope_auc.round(3))

# ── 4) 셔터 1만 쓰면 빠지는 정보 ──────────────────────────────────────
# (a) 장비별 셔터 1 시간 비율: 평소 vs 고장 24시간 전(종류 무관)
is_any_prefault_all = np.logical_or.reduce([prefault_all[fault] for fault in FAULTS])
tool_of_row_all = df_all.Tool.astype(str).to_numpy()
def shutter1_time_share(row_mask):
    return (pd.Series(all_weight * is_etching_all * row_mask).groupby(tool_of_row_all).sum()
            / pd.Series(all_weight * row_mask).groupby(tool_of_row_all).sum() * 100)
shutter1_by_tool = pd.DataFrame({'평소': shutter1_time_share(is_normal_all),
                                 '고장 24시간 전 (종류 무관)': shutter1_time_share(is_any_prefault_all)}).round(1)
fig = go.Figure()
for column_name, color in zip(shutter1_by_tool.columns, [C_OK, C_FAIL]):
    fig.add_bar(x=shutter1_by_tool.index, y=shutter1_by_tool[column_name], name=column_name, marker_color=color)
fig.update_layout(title='그림 2-3b. 장비별 셔터 1(식각 중) 시간 비율 — 평소 vs 고장 24시간 전', yaxis_title='셔터 1 비율(%)', barmode='group')
show(fig, 420)

# (b) 고장 직전 기록 유무: 모든 기록 기준 vs 식각(셔터 1) 기록 기준
def no_record_share(segment_mask, window_sec):
    # 고장 기록마다 직전 window_sec 안에 segment_mask 구간이 하나도 없는 비율(%)
    no_record = {fault: [] for fault in FAULTS}
    for tool, row_idx in df_all.groupby('Tool', observed=True).indices.items():
        row_idx = row_idx[segment_mask[row_idx]]
        seg_start, seg_end = df_all.time_start.to_numpy()[row_idx], df_all.time_end.to_numpy()[row_idx]
        longest = (seg_end - seg_start).max() if len(row_idx) else 0
        for fault in events[events.Tool == tool].itertuples():
            lo = np.searchsorted(seg_start, fault.time - window_sec - longest, 'left')
            hi = np.searchsorted(seg_start, fault.time, 'left')
            overlaps = (seg_end[lo:hi] > fault.time - window_sec).any()
            no_record[fault.fault_type].append(not overlaps)
    return {FSHORT[fault]: round(np.mean(values) * 100, 1) for fault, values in no_record.items()}

record_table = pd.DataFrame({
    '직전 1시간 기록 없음 · 모든 기록(%)': no_record_share(all_rows, HOUR),
    '직전 1시간 기록 없음 · 식각 기록(%)': no_record_share(is_etching_all, HOUR),
    '직전 24시간 기록 없음 · 모든 기록(%)': no_record_share(all_rows, DAY),
    '직전 24시간 기록 없음 · 식각 기록(%)': no_record_share(is_etching_all, DAY)}).reindex(ORDER)
record_table.index.name = '고장'
display(record_table)

# ── 이후 분석 대상: 셔터 1 구간만 ────────────────────────────────────
# step_s1/ = 원본 4초 행에서 셔터 1만 먼저 거른 뒤 스텝 구간으로 요약한 데이터 (src/make_processed.py)
df = pd.concat([pd.read_parquet(path, columns=load_cols) for path in sorted((PROC / 'step_s1').glob('*.parquet'))],
               ignore_index=True)
df['Tool'] = df.Tool.astype('category')
df['M'] = df.Tool.astype(str).str[:3]
df = df.sort_values(['Tool', 'time_start']).reset_index(drop=True)
assert len(df) == sum(tool_info['step_s1_rows'] for tool_info in manifest.values())

# 옛 방식(전체를 스텝 구간으로 묶은 뒤 첫 셔터 값이 1인 구간)과 비교
mix_total = pd.DataFrame([tool_info['shutter_mix'] for tool_info in manifest.values()]).sum()
old_filtered = df_all[df_all.shutter == 1]
print(f'스텝 구간 안에서 셔터 값이 바뀌는 구간: {mix_total.mixed_segments:,}개 / {mix_total.step_segments:,}개 · '
      f'옛 방식에 섞인 셔터 1 아닌 원본 행: {mix_total.old_rows_not_etching:,}행')
print(f'구간 수: 새 방식 {len(df):,} · 옛 방식 {len(old_filtered):,} / '
      f'원본 행: 새 방식 {df.n_rows.sum():,} · 옛 방식 {old_filtered.n_rows.sum():,}')
is_shutter1 = np.ones(len(df), bool)                       # 3장 이후 df는 모두 셔터 1 (기존 코드 호환용)
print(f'3장 이후 분석 대상: 셔터 1 구간 {len(df):,}개 (전체의 {len(df) / len(df_all):.1%}) · '
      f'시간 기준 {df.n_rows.sum() / RAW_ROWS:.1%}')

,전체 · 구간 수,전체 · 시간,압력 저하 24시간 전 · 시간,압력 과다 24시간 전 · 시간,누설 24시간 전 · 시간
셔터 값 (%),,,,,
결측,0.0,0.0,0.0,0.0,0.0
0,35.2,35.9,39.9,47.3,36.0
1 (식각 중),21.6,61.0,56.8,48.6,61.5
2,21.5,2.1,2.1,1.7,1.3
3,21.6,0.8,0.9,1.2,0.8
255,0.1,0.1,0.2,1.3,0.4


구간 수: 식각 중 526,266 · 셔터 0 857,518


,식각 중 중앙값,셔터 0 중앙값,차이(표준편차 단위)
센서,,,
챔버 압력,0.55,-0.65,1.97
아르곤 유량(PBN),0.59,-0.53,1.94
억제 전류,0.71,-0.79,1.83
빔 전류,0.83,-0.77,1.80
억제 전압,0.41,-0.65,1.76
아르곤 유량(소스),0.65,-0.48,1.59
빔 전압,0.77,-0.69,1.51
냉각 압력,0.34,-0.21,1.50
냉각 유량,0.53,-0.14,1.48


**장비별 AUC 중앙값** (0.5 = 구분 못 함, 0.5에서 멀수록 잘 구분)

범위           전체 구간   셔터 1  |AUC−0.5| 전체  |AUC−0.5| 셔터 1
센서    고장                                               
냉각 압력 압력 저하  0.397  0.365         0.103           0.135
      압력 과다  0.529  0.539         0.029           0.039
      누설     0.346  0.291         0.154           0.209
냉각 유량 압력 저하  0.494  0.473         0.006           0.027
      압력 과다  0.486  0.455         0.014           0.045
      누설     0.515  0.494         0.015           0.006

,직전 1시간 기록 없음 · 모든 기록(%),직전 1시간 기록 없음 · 식각 기록(%),직전 24시간 기록 없음 · 모든 기록(%),직전 24시간 기록 없음 · 식각 기록(%)
고장,,,,
압력 저하,31.6,37.0,0.9,1.1
압력 과다,59.8,70.6,29.7,31.2
누설,29.9,43.8,4.4,6.6


스텝 구간 안에서 셔터 값이 바뀌는 구간: 33개 / 2,437,615개 · 옛 방식에 섞인 셔터 1 아닌 원본 행: 703행
구간 수: 새 방식 526,268 · 옛 방식 526,266 / 원본 행: 새 방식 50,132,320 · 옛 방식 50,132,982
3장 이후 분석 대상: 셔터 1 구간 526,268개 (전체의 21.6%) · 시간 기준 61.0%


**해석**

- **셔터 1은 구간 수로는 21.6%지만 시간으로는 61.0%다.** 셔터 2·3은 구간 수는 많지만 짧은 전환 구간이다(시간 2.1%·0.8%). 셔터 0은 시간의 35.9%다.
- **식각 중과 셔터 0은 사실상 다른 상태다.** 공정 센서 10개가 표준편차 1.4~2.0배 차이 난다(냉각 압력 1.50, 냉각 유량 1.48). 두 상태를 섞으면 센서 흔들림의 상당 부분이 "셔터가 열렸나 닫혔나"가 된다. 회전·단계 소요시간·카운터는 차이가 없다.
- **셔터 1만 보면 냉각 압력의 구분력이 오른다.** |AUC−0.5|: 누설 0.154 → 0.209, 압력 저하 0.103 → 0.135, 압력 과다 0.029 → 0.039. 냉각 유량은 압력 과다에서 0.014 → 0.045로 오르고 나머지는 0.5 근처 그대로다.

**셔터 1만 쓰면 빠지는 정보**

| 정보 | 전체 | 셔터 1만 | 의미 |
|---|---|---|---|
| 압력 과다 24시간 전, 셔터 0 시간 비율 | 평균 35.9% | **47.3%** | 압력 과다 전에는 멈춰 있거나 대기하는 시간이 늘어난다. 셔터 1만 보면 이 힌트가 사라진다 |
| 직전 1시간 기록 없음 (저하 / 과다 / 누설) | 31.6 / 59.8 / 29.9% | **37.0 / 70.6 / 43.8%** | 식각 기록만 보면 직전 정보가 더 비는 사건이 5~14%p 늘어난다 |
| 직전 24시간 기록 없음 | 0.9 / 29.7 / 4.4% | 1.1 / 31.2 / 6.6% | 24시간 단위에서는 차이가 작다 |

**거르는 순서는 결과에 거의 영향이 없다.** 스텝 구간 안에서 셔터 값이 바뀌는 구간은 2,437,615개 중 33개뿐이다. 전체를 먼저 묶고 첫 셔터 값으로 거르면 셔터 1이 아닌 원본 행 703개가 섞이는데(셔터 1 원본 5,013만 행의 0.001%), 셔터 1만 먼저 거르면 이것이 없다. 그래도 정확한 쪽인 **셔터 1 먼저 거르기**(`step_s1/`)를 쓴다.

**결정**: 3장부터는 **셔터 1 구간만** 본다. 센서 비교가 공정하고 냉각 압력 신호가 더 뚜렷하기 때문이다. 대신 압력 과다 전 "셔터 0 시간 증가"와 직전 기록은 셔터 1만으로는 볼 수 없다. 모델에서 셔터 0 시간 비율을 별도 입력으로 넣을지는 비교해 볼 만하다.

## 3. 전처리·데이터 품질

### 3-1. 결측·구간 길이·구간 사이 공백

- 센서 결측과 정보 열 결측을 본다. TTF 결측은 오류가 아니다.
- 구간 길이는 제각각이라, **구간 수 기준 비율과 시간(원본 행) 기준 비율이 다르다.** 이후 비율은 필요한 곳에서 둘 다 적는다.
- 같은 장비에서 앞 구간의 끝과 다음 구간의 시작 사이가 길면 기록이 끊긴 것이다.

In [5]:
# ── 결측률: TTF 결측은 "기간 안에 고장 없음"이라 제외하고 본다 ──
missing_rate = (df.drop(columns=list(TTF.values())).isna().mean() * 100).round(3)
missing_rate = missing_rate[missing_rate > 0].rename('결측률(%)')
display(missing_rate.to_frame() if len(missing_rate) else pd.DataFrame({'결측률(%)': []}))

# ── 구간 길이 분포 (구간 수 기준 분위수) ─────────────────────────
length_quantiles = [.1, .25, .5, .75, .9, .99]
segment_length = pd.DataFrame({'지속시간(초)': df.duration_s.quantile(length_quantiles),
                               '원본 행 수': df.n_rows.quantile(length_quantiles)}).round(0)
segment_length.index = [f'{int(quantile * 100)}%' for quantile in length_quantiles]
display(segment_length)
print(f'원본 1행짜리 구간: {(df.n_rows == 1).mean():.1%} · 1시간 넘는 구간: {(df.duration_s > HOUR).mean():.2%} '
      f'(원본 행 기준 {df.n_rows[df.duration_s > HOUR].sum() / df.n_rows.sum():.1%})')

# ── 구간 사이 공백: 같은 장비에서 다음 구간 시작 − 이번 구간 끝 ──────
gap_between_segments = (df.groupby('Tool', observed=True).time_start.shift(-1) - df.time_end).dropna()
print(f'구간 사이 공백: 60초 초과 {(gap_between_segments > 60).mean():.1%} · '
      f'1시간 초과 {(gap_between_segments > HOUR).sum():,}곳 · 1일 초과 {(gap_between_segments > DAY).sum():,}곳')

fig = px.histogram(x=np.log10(df.duration_s.clip(lower=1)), nbins=60, color_discrete_sequence=[C_OK],
                   title='그림 3-1. 스텝 구간 지속시간 분포 (로그 눈금)')
fig.update_xaxes(tickvals=np.log10([1, 10, 60, 600, 3600, 36000]), ticktext=['1초', '10초', '1분', '10분', '1시간', '10시간'],
                 title_text='지속시간')
fig.update_yaxes(title_text='구간 수')
show(fig, 380)

,결측률(%)


,지속시간(초),원본 행 수
10%,44.0,12.0
25%,96.0,25.0
50%,296.0,75.0
75%,476.0,120.0
90%,828.0,208.0
99%,3064.0,766.0


원본 1행짜리 구간: 0.1% · 1시간 넘는 구간: 0.74% (원본 행 기준 7.5%)
구간 사이 공백: 60초 초과 69.0% · 1시간 초과 25,301곳 · 1일 초과 354곳


### 3-2. 평소 기준·고장 전 라벨·데이터 끝 문제

- **평소**: 가장 가까운 고장 기록에서 24시간 넘게 떨어진 구간. 정상으로 확정된 라벨은 아니다.
- **평소 기준(z값)**: 같은 **장비·레시피·스텝·셔터** 조합의 평소 구간에서 평균·표준편차를 구해 표준화한다. 평소 구간이 30개 미만인 조합은 z값을 비운다.
- **고장 전**: 고장 종류별로 TTF(구간 안 최솟값) ≤ 24시간.
- **데이터 끝**: TTF 결측은 "기록 기간 안에 고장 없음"이다. 장비 기록이 끝나기 직전 24시간은 그 뒤에 고장이 났는지 알 수 없는데, 결측이면 평소처럼 보인다.

In [6]:
# ── 평소: 가장 가까운 고장 기록에서 24시간 넘게 떨어진 구간 ─────────
df['fault_dist'] = nearest_fault_dist(df, events)
is_normal = (df.fault_dist > DAY).to_numpy()

# ── 조건 묶음 = 장비·레시피·스텝·셔터 (결측은 -1로 하나의 값처럼 취급) ──
df['_rs'] = df.recipe_step.fillna(-1)
df['_sh'] = df.shutter.fillna(-1)
CTX_KEYS = ['Tool', 'recipe', '_rs', '_sh']
df['ctx'] = df.groupby(CTX_KEYS, observed=True, sort=False).ngroup()

# 조건 묶음별 평소 평균·표준편차 (구간 하나를 1개로 셈, 가중치 없음)
normal_segments = df.loc[is_normal]
normal_count = normal_segments.groupby('ctx').size()
normal_stats = normal_segments.groupby('ctx')[FEAT].agg(['mean', 'std'])
has_enough_normal = normal_count.reindex(normal_stats.index).ge(30)      # 평소 구간 30개 미만이면 기준 없음
ctx_id = df.ctx.to_numpy()

# z값 = (구간 평균 − 그 조건의 평소 평균) / 평소 표준편차
z_by_sensor = {}
for sensor, mean_col in zip(SENSORS, FEAT):
    normal_mean = normal_stats[(mean_col, 'mean')].reindex(ctx_id).to_numpy()
    normal_std = (normal_stats[(mean_col, 'std')]
                  .where(has_enough_normal & (normal_stats[(mean_col, 'std')] > 1e-6))   # 표준편차 0이면 비움
                  .reindex(ctx_id).to_numpy())
    z_by_sensor[sensor] = ((df[mean_col].to_numpy() - normal_mean) / normal_std).astype('float32')

# ── 고장 전 라벨과 판단 불가 구간 ─────────────────────────────────
is_prefault = {fault: (df[TTF[fault]] <= DAY).to_numpy() for fault in FAULTS}   # 고장 종류별 고장 전 24시간
tool_end_time = df.groupby('Tool', observed=True).time_end.transform('max')
has_no_ttf = df[list(TTF.values())].isna().all(axis=1)
# 장비 기록 끝 24시간 안이면서 TTF가 모두 결측: 그 뒤에 고장이 났는지 알 수 없음
is_censored = ((tool_end_time - df.time_end) < DAY) & has_no_ttf

# ── 구분별 구간 수·시간 비율 표 ───────────────────────────────────
seg_weight = df.n_rows.to_numpy()      # 가중치 = 구간의 원본 행 수(구간 길이) → "원본 행 비율"은 시간 기준
label_rows = [{'구분': '평소 (고장에서 24시간 넘게 떨어짐)', '구간 수': int(is_normal.sum()),
               '원본 행 비율(%)': round(seg_weight[is_normal].sum() / seg_weight.sum() * 100, 1),
               '냉각 압력 z 있음(%)': round(np.isfinite(z_by_sensor['FLOWCOOLPRESSURE'][is_normal]).mean() * 100, 1)}]
for fault in FAULTS:
    label_rows.append({'구분': f'{FSHORT[fault]} 24시간 전', '구간 수': int(is_prefault[fault].sum()),
                       '원본 행 비율(%)': round(seg_weight[is_prefault[fault]].sum() / seg_weight.sum() * 100, 1),
                       '냉각 압력 z 있음(%)': round(np.isfinite(z_by_sensor['FLOWCOOLPRESSURE'][is_prefault[fault]]).mean() * 100, 1)})
label_rows.append({'구분': '데이터 끝 24시간 · TTF 모두 결측 (판단 불가)', '구간 수': int(is_censored.sum()),
                   '원본 행 비율(%)': round(seg_weight[is_censored.to_numpy()].sum() / seg_weight.sum() * 100, 1),
                   '냉각 압력 z 있음(%)': np.nan})
display(pd.DataFrame(label_rows).set_index('구분'))
print(f'조건(장비·레시피·스텝·셔터) 조합 {df.ctx.nunique():,}개 중 평소 구간 30개 이상: {int(has_enough_normal.sum()):,}개')

,구간 수,원본 행 비율(%),냉각 압력 z 있음(%)
구분,,,
평소 (고장에서 24시간 넘게 떨어짐),463670,88.2,97.9
압력 저하 24시간 전,31340,5.6,97.4
압력 과다 24시간 전,7033,1.5,97.3
누설 24시간 전,3872,0.9,97.1
데이터 끝 24시간 · TTF 모두 결측 (판단 불가),1214,0.2,NaN


조건(장비·레시피·스텝·셔터) 조합 3,110개 중 평소 구간 30개 이상: 598개


### 3-3. 중복·이상치

**중복**은 원본 행, 고장 기록, 스텝 구간 세 단계에서 본다. 가공(`src/make_processed.py`)은 값까지 완전히 같은 행만 지우고, 같은 시각에 값이 다른 행은 남겼다.

**이상치**는 센서 구간 평균으로 본다. 센서 값은 20대 전체 기준으로 표준화돼 있어 |값|은 전체 표준편차의 몇 배인지를 뜻한다. 기준은 두 가지다.
- |값| > 5 (전체 표준편차 5배 밖)
- 사분위 범위(IQR)의 3배 밖: 25% 값 − 3×IQR 보다 작거나 75% 값 + 3×IQR 보다 큼

극단값이 **측정 오류인지 고장 신호인지** 가리려고, 극단값 구간 중 고장 24시간 전(종류 무관)인 비율을 전체 평균과 비교한다.

In [7]:
import pyarrow.csv as pacsv

# ── 1) 중복 ──────────────────────────────────────────────────────
# 원본 완전 중복: 가공 때 지운 수 (manifest 기록)
exact_dup_rows = sum(tool_info['exact_duplicates_dropped'] for tool_info in manifest.values())
raw_rows_total = sum(tool_info['rows_after_dedup'] for tool_info in manifest.values()) + exact_dup_rows

# 원본 같은 시각 중복: 원본 CSV가 있을 때만 time 열만 읽어 센다 (원본은 저장소에 없음)
raw_files = sorted((ROOT / 'data/raw/phm_data_challenge_2018/train').glob('*_DC_train.csv'))
same_time_rows = np.nan
if raw_files:
    same_time_rows = 0
    for raw_path in raw_files:
        raw_times = pacsv.read_csv(raw_path, convert_options=pacsv.ConvertOptions(include_columns=['time'])).column('time').to_numpy()
        same_time_rows += int((np.diff(np.sort(raw_times)) == 0).sum())      # 앞 행과 시각이 같은 행 수

fault_exact_dup = int(faults.duplicated(['Tool', 'time', 'fault_name']).sum())   # 장비·시각·고장 이름이 모두 같음
cross_type_same_time = int(events.duplicated(['Tool', 'time']).sum())             # 같은 시각에 다른 종류 고장
next_seg_start = df.groupby('Tool', observed=True).time_start.shift(-1)
seg_overlap = int((next_seg_start < df.time_end).sum())                           # 다음 구간과 시간이 겹침
seg_same_start = int(df.duplicated(['Tool', 'time_start']).sum())                  # 시작 시각이 같은 구간

dup_table = pd.DataFrame([
    {'단계': '원본', '항목': '완전 중복 행 (값까지 같음)', '개수': exact_dup_rows,
     '비율(%)': exact_dup_rows / raw_rows_total * 100, '처리': '가공 때 제거'},
    {'단계': '원본', '항목': '같은 시각 중복 행 (완전 중복 제외, 값이 다름)', '개수': same_time_rows - exact_dup_rows,
     '비율(%)': (same_time_rows - exact_dup_rows) / raw_rows_total * 100, '처리': '남김 (어느 값이 맞는지 알 수 없음)'},
    {'단계': '고장 기록', '항목': '장비·시각·이름이 같은 중복', '개수': fault_exact_dup,
     '비율(%)': fault_exact_dup / len(faults) * 100, '처리': '제거 (1,236 → 1,211)'},
    {'단계': '고장 기록', '항목': '같은 시각에 다른 종류 고장', '개수': cross_type_same_time,
     '비율(%)': cross_type_same_time / len(events) * 100, '처리': '-'},
    {'단계': '스텝 구간', '항목': '다음 구간과 시간이 겹침', '개수': seg_overlap, '비율(%)': seg_overlap / len(df) * 100, '처리': '-'},
    {'단계': '스텝 구간', '항목': '시작 시각이 같은 구간', '개수': seg_same_start,
     '비율(%)': seg_same_start / len(df) * 100, '처리': '남김 (같은 시각 중복 행에서 생김)'},
]).set_index(['단계', '항목'])
display(dup_table.round(4))

# ── 2) 센서 극단값 (구간 평균, 구간 하나를 1개로 셈) ───────────────────
outlier_rows = []
for sensor, mean_col in zip(SENSORS, FEAT):
    sensor_values = df[mean_col]
    q25, q75 = sensor_values.quantile([.25, .75])
    iqr = q75 - q25
    outside_iqr = ((sensor_values < q25 - 3 * iqr) | (sensor_values > q75 + 3 * iqr)).mean() * 100 if iqr > 0 else np.nan
    outlier_rows.append({'센서': NAMES[sensor], '|값|>5 (%)': (sensor_values.abs() > 5).mean() * 100,
                         '|값|>10 (%)': (sensor_values.abs() > 10).mean() * 100, 'IQR×3 밖 (%)': outside_iqr,
                         '최솟값': sensor_values.min(), '1%': sensor_values.quantile(.01),
                         '99%': sensor_values.quantile(.99), '최댓값': sensor_values.max()})
outlier_table = pd.DataFrame(outlier_rows).set_index('센서').sort_values('|값|>5 (%)', ascending=False)
display(outlier_table.round(3))

# ── 3) 극단값이 고장 근처에 몰려 있는가 ────────────────────────────────
hours_to_nearest_next_fault = df[list(TTF.values())].min(axis=1)                 # 다음 고장(종류 무관)까지 남은 시간(초)
is_any_prefault_24h = (hours_to_nearest_next_fault <= DAY).to_numpy()
pressure_high_prefault = (df[TTF[FAULTS[1]]] <= DAY).to_numpy()
extreme_checks = [('냉각 압력 > 5', df.FLOWCOOLPRESSURE_mean > 5),
                  ('냉각 압력 < −3', df.FLOWCOOLPRESSURE_mean < -3),
                  ('챔버 압력 |값| > 5', df.IONGAUGEPRESSURE_mean.abs() > 5),
                  ('회전 속도 |값| > 10', df.ROTATIONSPEED_mean.abs() > 10)]
near_fault_rows = []
for check_name, extreme_mask in extreme_checks:
    extreme_mask = extreme_mask.to_numpy()
    near_fault_rows.append({'극단값': check_name, '구간 수': int(extreme_mask.sum()),
                            '나온 장비 수': df.loc[extreme_mask, 'Tool'].nunique(),
                            '고장 24시간 전 비율(%)': is_any_prefault_24h[extreme_mask].mean() * 100,
                            '그중 압력 과다 24시간 전(%)': pressure_high_prefault[extreme_mask].mean() * 100})
near_fault_table = pd.DataFrame(near_fault_rows).set_index('극단값')
display(near_fault_table.round(1))
print(f'비교 기준: 전체 구간 중 고장 24시간 전 비율 {is_any_prefault_24h.mean():.1%}')

# ── 4) 누적 카운터 되감김, 아주 긴 구간 ────────────────────────────────
counter_rows = []
for counter in COUNTERS:
    counter_change = df.groupby('Tool', observed=True)[counter + '_mean'].diff()   # 같은 장비에서 다음 구간 − 이번 구간
    big_drop = counter_change < -0.5
    drops_per_tool = df[big_drop].groupby('Tool', observed=True).size().reindex(df.Tool.cat.categories, fill_value=0)
    counter_rows.append({'카운터': NAMES[counter], '감소 횟수': int((counter_change < 0).sum()),
                         '0.5 넘게 감소': int(big_drop.sum()), '장비당 큰 감소 중앙값': drops_per_tool.median()})
display(pd.DataFrame(counter_rows).set_index('카운터'))
print(f'10시간 넘는 구간 {(df.duration_s > 10 * HOUR).sum():,}개 · 최장 {df.duration_s.max() / HOUR:.1f}시간')

# 그림 3-3: 센서별 극단값 비율
bar_data = outlier_table.sort_values('IQR×3 밖 (%)')
fig = go.Figure()
fig.add_bar(y=bar_data.index, x=bar_data['IQR×3 밖 (%)'], orientation='h', name='IQR×3 밖', marker_color=C_GREY)
fig.add_bar(y=bar_data.index, x=bar_data['|값|>5 (%)'], orientation='h', name='|값| > 5', marker_color=C_FAIL)
fig.update_layout(title='그림 3-3. 센서별 극단값 비율 (구간 평균 기준)', barmode='group', xaxis_title='구간 비율(%)',
                  legend=dict(orientation='h', y=-0.15))
show(fig, 560)

개수   비율(%)                     처리
단계    항목                                                                 
원본    완전 중복 행 (값까지 같음)                5470  0.0067                가공 때 제거
      같은 시각 중복 행 (완전 중복 제외, 값이 다름)  112063  0.1363  남김 (어느 값이 맞는지 알 수 없음)
고장 기록 장비·시각·이름이 같은 중복                   25  2.0227     제거 (1,236 → 1,211)
      같은 시각에 다른 종류 고장                    0  0.0000                      -
스텝 구간 다음 구간과 시간이 겹침                      0  0.0000                      -
      시작 시각이 같은 구간                       9  0.0017   남김 (같은 시각 중복 행에서 생김)

,|값|>5 (%),|값|>10 (%),IQR×3 밖 (%),최솟값,1%,99%,최댓값
센서,,,,,,,
회전 각도,1.379,0.008,5.036,-27.650999,-0.331,6.156,16.652000
단계 소요시간,0.341,0.016,2.065,-1.186000,-0.968,4.038,24.242001
회전 속도,0.225,0.225,11.353,-12.818000,-0.512,2.111,25.677999
냉각 압력,0.009,0.005,0.023,-3.036000,-0.190,0.574,16.618000
억제 전류,0.003,0.002,0.017,-1.515000,-0.939,1.810,23.702999
챔버 압력,0.002,0.000,0.075,-1.908000,-0.221,1.144,9.770000
억제 전압,0.000,0.000,0.000,-1.575000,-0.977,1.602,1.821000
빔 전압,0.000,0.000,0.000,-1.529000,-0.974,1.546,1.803000
아르곤 유량(소스),0.000,0.000,2.396,-1.901000,-1.266,1.499,2.128000


,구간 수,나온 장비 수,고장 24시간 전 비율(%),그중 압력 과다 24시간 전(%)
극단값,,,,
냉각 압력 > 5,48,15,27.1,27.1
냉각 압력 < −3,1,1,0.0,0.0
챔버 압력 |값| > 5,9,2,0.0,0.0
회전 속도 |값| > 10,1182,8,3.4,0.8


비교 기준: 전체 구간 중 고장 24시간 전 비율 7.3%


,감소 횟수,0.5 넘게 감소,장비당 큰 감소 중앙값
카운터,,,
그리드 사용량,133,126,7.0
실드 사용량1,108,69,3.0
실드 사용량2,192,138,6.5


10시간 넘는 구간 2개 · 최장 42.1시간


**해석**
- **중복은 아주 적다.** 원본 완전 중복은 0.007%(5,470행), 같은 시각에 값이 다른 행은 0.14%다. 고장 기록 중복 25건(2%)은 같은 고장이 두 번 찍힌 것이다. 스텝 구간끼리 시간이 겹치는 경우는 없다.
- **냉각 압력의 큰 값(> 5)은 오류가 아니라 고장 신호다.** 48구간 중 27.1%가 고장 24시간 전이고, 전부 압력 과다 전이다(셔터 1 평균 7.3%의 약 4배). 장비 15대에 나온다.
- **냉각 압력의 작은 값(< −3)은 셔터 1에서는 1구간뿐이다.** 전체 구간에서 보이던 한 장비의 낮은 수준은 대부분 셔터 1 밖이었다.
- **회전 속도·회전 각도는 극단값이 많다.** 회전 속도는 −12.8 ~ +25.7, IQR×3 밖이 11.4%다. |값| > 10 구간(1,182개, 장비 8대)은 고장과 무관하다(3.4%). 다른 운전 모드나 측정 방식 차이로 보인다.
- **챔버 압력 최댓값 9.77**: |값| > 5는 9구간(장비 2대)뿐인 순간 튐이다. 고장과 무관하다(0%).
- 공정 센서 대부분(빔 전압·전류, 억제 전압, 아르곤, 카운터)은 |값| > 5가 없다.
- 누적 카운터는 장비당 3~7번 크게 줄어든다. 정비 때 초기화된 것으로 보인다.

**처리 방안**

| 대상 | 처리 | 이유 |
|---|---|---|
| 원본 완전 중복 행 | **제거** (가공 때 완료) | 같은 기록이 두 번 저장된 것 |
| 원본 같은 시각·다른 값 행 | **남김** (스텝 구간 평균에 함께 들어감) | 어느 값이 맞는지 알 수 없고 0.14%라 영향이 작다 |
| 고장 기록 중복 | **제거** (1,236 → 1,211) | 같은 고장이 두 번 찍힌 것. 사건 수를 부풀린다 |
| 냉각 압력 큰 값(> 5) | **남김** | 압력 과다의 신호다. 지우면 예측할 정보를 버리게 된다 |
| 냉각 압력 작은 값(< −3) | **남김**, 장비별 z값으로 흡수 | 셔터 1에서는 1구간뿐이다. 장비·레시피·스텝·셔터별 기준이면 0 근처가 된다 |
| 회전 속도·회전 각도 | **입력에서 제외** | 극단값이 많고 고장과 무관하다 |
| 챔버 압력 순간 튐 | **남김** (모델에서는 z값을 ±10에서 자르는 방식 검토) | 9구간뿐이라 영향이 작다 |
| 누적 카운터 | **모델에서 넣은 경우·뺀 경우 비교** | 장비당 3~7번 초기화된다. 같은 장비 안 구분력은 높지만(4-5) 노화 효과인지 확인이 필요하다 |
| 10시간 넘는 구간(셔터 1에서 2개, 최장 42.1시간) | **남김** | 장비가 멈춘 시간으로 보인다. 2개뿐이라 영향이 작다 |

# **정리**: 지워야 할 이상치는 없다. 극단값이 많은 회전 컬럼은 입력에서 빼고, 나머지는 z값(장비·공정별 평소 기준)이 흡수한다.

## 4. 데이터 시각화·상세 분석

### 4-1. 장비별 고장 분포 (Q5)

고장이 장비에 몰려 있으면 같은 장비가 학습·검증에 섞일 때 점수가 부풀려진다.

In [8]:
# 장비별 고장 기록 수와 누적 비중 (상위 5대)
faults_per_tool = tool_scale['고장 합계'].sort_values(ascending=False)
fault_share = (faults_per_tool / faults_per_tool.sum() * 100).round(1)
display(pd.DataFrame({'고장 기록': faults_per_tool, '비중(%)': fault_share, '누적(%)': fault_share.cumsum().round(1)}).head(5))

# 그림 4-1: 장비별·고장 종류별 쌓은 막대 (고장 많은 장비부터)
fault_counts_long = fault_counts_by_type.reset_index().melt(id_vars='Tool', var_name='고장 종류', value_name='건수')
fig = px.bar(fault_counts_long, x='Tool', y='건수', color='고장 종류', color_discrete_map=FCOLOR,
             category_orders={'Tool': list(faults_per_tool.index), '고장 종류': ORDER},
             title='그림 4-1. 장비별 고장 기록 (중복 제외 1,211건)')
show(fig, 420)

,고장 기록,비중(%),누적(%)
Tool,,,
M02_03,356,29.4,29.4
M01_09,150,12.4,41.8
M01_08,117,9.7,51.5
M02_01,109,9.0,60.5
M01_03,91,7.5,68.0


### 4-2. 레시피·스텝이 센서 값에 주는 영향 (Q1)

센서 값의 전체 분산 중 각 기준으로 묶었을 때 설명되는 비율(η², 0~1)이다. 셔터 1 구간만 쓰므로 셔터 효과는 이미 빠져 있다. 1분 요약 기준(모든 셔터)으로는 스텝이 약 44%를 설명했다. 이 값이 크면 센서 값을 그대로 비교하면 안 되고, 같은 스텝의 평소 값과 비교해야 한다.

In [9]:
def eta_squared(group_keys):
    """센서별 η² = 1 − (묶음 안 분산 / 전체 분산). 그 기준으로 묶었을 때 센서 흔들림을 얼마나 설명하는지(0~1)."""
    explained = {}
    for sensor in SENSORS:
        mean_col = sensor + '_mean'
        valid_rows = df[group_keys + [mean_col]].dropna()
        values = valid_rows[mean_col].astype('float64')
        group_mean = valid_rows.groupby(group_keys, observed=True)[mean_col].transform('mean')
        within_var = (values - group_mean).pow(2).mean()                 # 묶음 평균에서 벗어난 정도
        explained[NAMES[sensor]] = 1 - within_var / values.var(ddof=0)
    return pd.Series(explained)

eta_table = pd.DataFrame({'장비': eta_squared(['Tool']), 'M01/M02': eta_squared(['M']), '레시피': eta_squared(['recipe']),
                          '스텝': eta_squared(['_rs']), '레시피+스텝': eta_squared(['recipe', '_rs']),
                          '장비+레시피+스텝+셔터': eta_squared(CTX_KEYS)})
eta_table = eta_table.sort_values('스텝', ascending=False).round(2)
display(eta_table)
display(eta_table.median().rename('중앙값').to_frame().T)
fig = px.imshow(eta_table, text_auto='.2f', color_continuous_scale='Blues', zmin=0, zmax=1, aspect='auto',
                title='그림 4-2. 센서 분산 중 각 기준이 설명하는 비율 (η²)')
show(fig, 560)

,장비,M01/M02,레시피,스텝,레시피+스텝,장비+레시피+스텝+셔터
냉각 유량,0.56,0.10,0.67,0.45,0.95,0.98
아르곤 유량(PBN),0.77,0.09,0.70,0.30,0.78,0.87
아르곤 유량(소스),0.05,0.00,0.45,0.25,0.93,0.98
웨이퍼 기울기,0.03,0.00,0.19,0.22,0.69,0.74
억제 전압,0.03,0.00,0.64,0.20,0.93,0.98
냉각 압력,0.52,0.03,0.34,0.18,0.61,0.90
회전 각도,0.02,0.00,0.55,0.17,0.84,0.91
억제 전류,0.07,0.00,0.53,0.14,0.79,0.87
챔버 압력,0.41,0.06,0.60,0.14,0.64,0.79
빔 전류,0.05,0.00,0.70,0.13,0.92,0.98


,장비,M01/M02,레시피,스텝,레시피+스텝,장비+레시피+스텝+셔터
중앙값,0.045,0.0,0.49,0.14,0.735,0.87


### 4-3. M01과 M02 (Q5)

M01·M02의 뜻은 공식 설명에 없다. 장비 쌍마다 **공유하는 레시피 수**를 본다. 같은 장비 안의 두 모듈이라면 같은 번호끼리 레시피를 공유할 것이고, 장비 종류가 다르다면 M01끼리·M02끼리만 공유할 것이다.

In [10]:
# 장비 쌍마다 공유하는 레시피 수
tool_names = list(df.Tool.cat.categories)
recipes_by_tool = {tool: set(df.loc[df.Tool == tool, 'recipe']) for tool in tool_names}
shared_recipes = pd.DataFrame([[len(recipes_by_tool[tool_a] & recipes_by_tool[tool_b]) if tool_a != tool_b else np.nan
                                for tool_b in tool_names] for tool_a in tool_names], index=tool_names, columns=tool_names)

# 쌍을 세 종류로 나눠 평균: 같은 기계 종류 / 다른 기계 종류 / 다른 종류지만 같은 번호
pairs = [(tool_a, tool_b) for tool_a in tool_names for tool_b in tool_names if tool_a < tool_b]
same_machine = [shared_recipes.loc[tool_a, tool_b] for tool_a, tool_b in pairs if tool_a[:3] == tool_b[:3]]
cross_machine = [shared_recipes.loc[tool_a, tool_b] for tool_a, tool_b in pairs if tool_a[:3] != tool_b[:3]]
same_number = [shared_recipes.loc[tool_a, tool_b] for tool_a, tool_b in pairs
               if tool_a[:3] != tool_b[:3] and tool_a[4:] == tool_b[4:]]
display(pd.DataFrame({'장비 쌍': ['M01끼리·M02끼리', 'M01–M02 사이', '같은 번호의 M01–M02'],
                      '쌍 수': [len(same_machine), len(cross_machine), len(same_number)],
                      '공유 레시피 평균': [round(np.mean(same_machine), 1), round(np.mean(cross_machine), 1),
                                     round(np.mean(same_number), 1)]}).set_index('장비 쌍'))
fig = px.imshow(shared_recipes, text_auto='.0f', color_continuous_scale='Blues', aspect='auto',
                title='그림 4-3. 장비 쌍별 공유 레시피 수')
show(fig, 640)

,쌍 수,공유 레시피 평균
장비 쌍,,
M01끼리·M02끼리,90,13.6
M01–M02 사이,100,0.1
같은 번호의 M01–M02,10,0.1


### 4-4. 셔터 위치 (Q5)

셔터 분석은 분석 대상을 정하는 근거라 **2-3**으로 옮겼다. 2-3 이후 모든 분석은 셔터 1(식각 중) 구간이다.

### 4-5. 센서별 고장 전·평소 구분력 (Q2)

각 장비 안에서 **고장 24시간 전 구간의 z값**과 **평소 구간의 z값**(장비당 최대 5만 구간 표본)을 비교한 AUC다. 0.5면 구분 못 함, 0.5에서 멀수록 잘 구분한다(0.5보다 작으면 고장 전에 값이 낮다는 뜻). 장비별 AUC의 **중앙값**을 그린다.

셔터 1 구간 결과다. 전체 구간과의 비교는 **2-3**에 있다. 구간 단위라 긴 구간과 짧은 구간이 같은 무게다.

In [11]:
auc_rng = np.random.default_rng(0)
rows_by_tool = df.groupby('Tool', observed=True).indices        # 장비별 행 번호 (이후 여러 절에서 사용)
auc_rows = []
for scope_name, scope_mask in [('셔터 1', is_shutter1)]:
    for tool, row_idx in rows_by_tool.items():
        normal_idx = row_idx[is_normal[row_idx] & scope_mask[row_idx]]
        if len(normal_idx) > 50_000:                              # 평소 구간은 장비당 최대 5만 개 표본
            normal_idx = auc_rng.choice(normal_idx, 50_000, replace=False)
        for fault in FAULTS:
            prefault_idx = row_idx[is_prefault[fault][row_idx] & scope_mask[row_idx]]
            for sensor in SENSORS:
                auc_rows.append({'범위': scope_name, 'Tool': tool, 'fault': FSHORT[fault], 'sensor': sensor,
                                 'auc': auc(z_by_sensor[sensor][prefault_idx], z_by_sensor[sensor][normal_idx])})
auc_df = pd.DataFrame(auc_rows)

# 장비별 AUC의 중앙값 (범위 × 센서 × 고장)
auc_median = auc_df.groupby(['범위', 'sensor', 'fault']).auc.median().unstack('fault')[ORDER]
auc_median_shutter1 = auc_median.loc['셔터 1'].reindex(SENSORS)
heatmap_data = auc_median_shutter1.copy()
heatmap_data.index = [NAMES[sensor] for sensor in heatmap_data.index]
fig = px.imshow(heatmap_data, text_auto='.3f', color_continuous_scale='RdBu_r', zmin=0.35, zmax=0.65, aspect='auto',
                title='그림 4-5. 장비별 AUC 중앙값 (셔터 1) — 0.5에서 멀수록 구분')
show(fig, 600)

# 센서별 구분력 = 고장 3종 중 0.5에서 가장 먼 값, 셔터 1과 전체 비교
max_deviation = (auc_median - .5).abs().max(axis=1).unstack('범위')[['셔터 1']]
max_deviation.index = [NAMES[sensor] for sensor in max_deviation.index]
n_tools_with_auc = (auc_df[auc_df.범위 == '셔터 1'].dropna()
                    .groupby(['sensor', 'fault']).Tool.nunique().unstack()[ORDER])
display(max_deviation.sort_values('셔터 1', ascending=False).round(3)
        .rename(columns=lambda scope_name: f'{scope_name} · |AUC−0.5| 최대'))
print('AUC를 계산한 장비 수(셔터 1, 냉각 압력):', n_tools_with_auc.loc['FLOWCOOLPRESSURE'].to_dict())

범위,셔터 1 · |AUC−0.5| 최대
냉각 압력,0.209
실드 사용량1,0.148
실드 사용량2,0.093
억제 전류,0.071
냉각 유량,0.045
아르곤 유량(소스),0.043
챔버 압력,0.038
빔 전압,0.032
회전 각도,0.027
아르곤 유량(PBN),0.027


AUC를 계산한 장비 수(셔터 1, 냉각 압력): {'압력 저하': 19, '압력 과다': 16, '누설': 14}


### 4-6. 고장 전 48시간 궤적 (Q3)

고장 기록마다 **고장 전 48시간**을 1시간 단위로 나눠, 셔터 1 구간의 z값을 원본 행 수로 가중 평균한다. 사건 사이의 **중앙값(선)과 사분위 범위(띠)**를 그린다. 0은 평소와 같다는 뜻이다.

In [12]:
TRAJ_SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE', 'IONGAUGEPRESSURE']
WINDOW_H = 48                                                  # 고장 전 몇 시간을 볼지
seg_start_all = df.time_start.to_numpy()

# ── 고장 기록마다 고장 전 48시간을 1시간 칸으로 나눠 가중 평균 ─────
traj_parts = []
for tool, row_idx in rows_by_tool.items():
    row_idx = row_idx[is_shutter1[row_idx]]                   # 셔터 1 구간만
    tool_start = seg_start_all[row_idx]
    tool_weight = seg_weight[row_idx]                          # 가중치 = 구간의 원본 행 수(구간 길이)
    tool_z = np.column_stack([z_by_sensor[sensor][row_idx] for sensor in TRAJ_SENSORS])
    for fault in events[events.Tool == tool].itertuples():
        win_lo = np.searchsorted(tool_start, fault.time - WINDOW_H * HOUR, 'left')
        win_hi = np.searchsorted(tool_start, fault.time, 'left')
        if win_hi <= win_lo:
            continue
        window = pd.DataFrame(tool_z[win_lo:win_hi], columns=TRAJ_SENSORS)
        window['h'] = (fault.time - tool_start[win_lo:win_hi] - 1) // HOUR   # 고장까지 몇 시간 전 칸 (0 = 직전 1시간)
        window['weight'] = tool_weight[win_lo:win_hi]
        hourly_mean = {}
        for sensor in TRAJ_SENSORS:
            has_z = window[sensor].notna()
            # 칸별 가중 평균 = Σ(z × weight) / Σ(weight)
            hourly_mean[sensor] = ((window.loc[has_z, sensor] * window.loc[has_z, 'weight']).groupby(window.loc[has_z, 'h']).sum()
                                   / window.loc[has_z, 'weight'].groupby(window.loc[has_z, 'h']).sum())
        event_traj = pd.DataFrame(hourly_mean).reset_index().rename(columns={'index': 'h'})
        event_traj['event'] = f'{tool}_{fault.time}_{fault.fault_type}'
        event_traj['fault_type'] = fault.fault_type
        traj_parts.append(event_traj)
traj = pd.concat(traj_parts, ignore_index=True)

# 사건 사이에서는 가중치 없이 중앙값과 사분위 범위 (사건 하나를 1개로 셈)
traj_profile = (traj.melt(id_vars=['event', 'fault_type', 'h'], value_vars=TRAJ_SENSORS, var_name='sensor', value_name='z')
                .dropna().groupby(['fault_type', 'sensor', 'h']).z
                .agg(median='median', q25=lambda z_values: z_values.quantile(.25),
                     q75=lambda z_values: z_values.quantile(.75), n_events='size').reset_index())

# 그림 4-6: 행 = 센서, 열 = 고장 종류
fig = make_subplots(rows=len(TRAJ_SENSORS), cols=3, shared_xaxes=True, vertical_spacing=.07,
                    subplot_titles=[f'{NAMES[sensor]} · {FSHORT[fault]}' for sensor in TRAJ_SENSORS for fault in FAULTS])
sensor_colors = dict(zip(TRAJ_SENSORS, [C_OK, C_FAIL, '#4c9a52']))
for row, sensor in enumerate(TRAJ_SENSORS, 1):
    for col, fault in enumerate(FAULTS, 1):
        profile = traj_profile[(traj_profile.sensor == sensor) & (traj_profile.fault_type == fault)].sort_values('h')
        x_hours = -profile.h.to_numpy() - 0.5                 # 칸 가운데를 음수 시간으로 (왼쪽이 먼 과거)
        fig.add_scatter(x=np.r_[x_hours, x_hours[::-1]], y=np.r_[profile.q75, profile.q25[::-1]], fill='toself', mode='none',
                        fillcolor='rgba(120,120,120,.18)', showlegend=False, row=row, col=col)        # 사분위 범위 띠
        fig.add_scatter(x=x_hours, y=profile['median'], mode='lines', line=dict(color=sensor_colors[sensor], width=2),
                        showlegend=False, hovertemplate='%{x:.1f}시간 전<br>중앙값 %{y:.3f}<extra></extra>', row=row, col=col)
        fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1), row=row, col=col)          # 0 = 평소 수준
fig.update_xaxes(title_text='고장까지 남은 시간(시간)', row=len(TRAJ_SENSORS))
fig.update_layout(title='그림 4-6. 고장 전 48시간 궤적 (셔터 1) — 사건 간 중앙값과 사분위 범위')
show(fig, 800)

# 직전 변화량 = (직전 3시간 중앙값) − (24~48시간 전 중앙값)
last_3h = traj_profile[traj_profile.h < 3].groupby(['fault_type', 'sensor'])['median'].median()
early_24_48h = traj_profile[(traj_profile.h >= 24) & (traj_profile.h < 48)].groupby(['fault_type', 'sensor'])['median'].median()
late_change = (last_3h - early_24_48h).rename('직전 3시간 − 24~48시간 전').reset_index()
late_change['고장 종류'] = late_change.fault_type.map(FSHORT)
late_change['센서'] = late_change.sensor.map(NAMES)
late_change['기록 있는 사건'] = late_change.fault_type.map(traj.groupby('fault_type').event.nunique())
display(late_change[['고장 종류', '센서', '직전 3시간 − 24~48시간 전', '기록 있는 사건']].round(3).reset_index(drop=True))

,고장 종류,센서,직전 3시간 − 24~48시간 전,기록 있는 사건
0,압력 저하,냉각 유량,-0.001,739
1,압력 저하,냉각 압력,0.293,739
2,압력 저하,챔버 압력,-0.029,739
3,압력 과다,냉각 유량,-0.108,265
4,압력 과다,냉각 압력,0.204,265
5,압력 과다,챔버 압력,-0.286,265
6,누설,냉각 유량,0.011,132
7,누설,냉각 압력,0.073,132
8,누설,챔버 압력,0.232,132


### 4-7. 고장 직전 모양 — 4초 해상도 (Q3)

스텝 구간도 평균이라 짧고 급한 변화가 묻힐 수 있다. TCS(2018)는 압력 저하는 냉각 압력이 **갑자기** 떨어지고, 압력 과다는 **서서히** 오른다고 보고했다. `prefault/` 원본(4초)에서 고장 기록마다 **직전 6시간**을 잘라 확인한다.

- 셔터 1 행만 쓴다.
- 3-2의 평소 기준(장비·레시피·스텝·셔터별 구간 평균의 평균·표준편차)으로 z값을 만든다. 기준이 구간 평균에서 나왔으므로 4초 값의 z는 더 크게 흔들린다.
- 사건마다 5분 단위 중앙값을 구한 뒤, 사건 사이 중앙값(선)과 사분위 범위(띠)를 그린다.

In [13]:
FINE_SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE']
PF_COLS = ['Tool', 'time', 'recipe', 'recipe_step', 'FIXTURESHUTTERPOSITION', *FINE_SENSORS]

# ── 고장 24시간 전 4초 원본(prefault/) 중 셔터 1 행만 ────────────────
prefault_raw = pd.concat([pd.read_parquet(path, columns=PF_COLS) for path in sorted((PROC / 'prefault').glob('*.parquet'))],
                         ignore_index=True)
prefault_etch = prefault_raw[prefault_raw.FIXTURESHUTTERPOSITION == 1].copy()
del prefault_raw                                               # 메모리 확보
prefault_etch['Tool'] = prefault_etch.Tool.map(label)
prefault_etch['_rs'] = prefault_etch.recipe_step.fillna(-1)
prefault_etch['_sh'] = 1.0

# 스텝 구간 데이터의 조건 묶음 번호(ctx)를 붙여, 같은 평소 기준으로 z값을 만든다
ctx_lookup = df[CTX_KEYS + ['ctx']].drop_duplicates(CTX_KEYS).copy()
ctx_lookup['Tool'] = ctx_lookup.Tool.astype(str)
prefault_etch = prefault_etch.merge(ctx_lookup, on=CTX_KEYS, how='left')
prefault_etch['ctx'] = prefault_etch.ctx.fillna(-1).astype(int)
for sensor in FINE_SENSORS:
    mean_col = sensor + '_mean'
    normal_mean = normal_stats[(mean_col, 'mean')].reindex(prefault_etch.ctx).to_numpy()
    normal_std = (normal_stats[(mean_col, 'std')]
                  .where(has_enough_normal & (normal_stats[(mean_col, 'std')] > 1e-6)).reindex(prefault_etch.ctx).to_numpy())
    prefault_etch[sensor + '_z'] = (prefault_etch[sensor].to_numpy() - normal_mean) / normal_std
prefault_etch = prefault_etch.sort_values(['Tool', 'time']).reset_index(drop=True)
print(f'고장 24시간 전 원본 중 셔터 1: {len(prefault_etch):,}행, 냉각 압력 z 계산됨: {prefault_etch.FLOWCOOLPRESSURE_z.notna().mean():.1%}')

# ── 고장 기록마다 직전 6시간을 5분 칸으로 나눠 중앙값 ─────────────────
FINE_WINDOW_SEC, BIN_SEC = 6 * HOUR, 300
FINE_Z_COLS = [sensor + '_z' for sensor in FINE_SENSORS]
fine_parts = []
prefault_rows_by_tool = prefault_etch.groupby('Tool').indices
for tool, row_idx in prefault_rows_by_tool.items():
    tool_time = prefault_etch.time.to_numpy()[row_idx]
    tool_z = prefault_etch[FINE_Z_COLS].to_numpy()[row_idx]
    for fault in events[events.Tool == tool].itertuples():
        win_lo = np.searchsorted(tool_time, fault.time - FINE_WINDOW_SEC, 'left')
        win_hi = np.searchsorted(tool_time, fault.time, 'left')
        if win_hi <= win_lo:
            continue
        window = pd.DataFrame(tool_z[win_lo:win_hi], columns=FINE_SENSORS)
        window['bin_5min'] = (fault.time - tool_time[win_lo:win_hi] - 1) // BIN_SEC   # 고장까지 몇 번째 5분 칸 (0 = 직전 5분)
        event_bins = window.groupby('bin_5min')[FINE_SENSORS].median().reset_index()
        event_bins['event'] = f'{tool}_{fault.time}_{fault.fault_type}'
        event_bins['fault_type'] = fault.fault_type
        fine_parts.append(event_bins)
fine_window = pd.concat(fine_parts, ignore_index=True)
fine_profile = (fine_window.melt(id_vars=['event', 'fault_type', 'bin_5min'], value_vars=FINE_SENSORS, var_name='sensor', value_name='z')
                .dropna().groupby(['fault_type', 'sensor', 'bin_5min']).z
                .agg(median='median', q25=lambda z_values: z_values.quantile(.25),
                     q75=lambda z_values: z_values.quantile(.75), n='size').reset_index())

# 그림 4-7a: 행 = 센서, 열 = 고장 종류
fig = make_subplots(rows=len(FINE_SENSORS), cols=3, shared_xaxes=True, vertical_spacing=.1,
                    subplot_titles=[f'{NAMES[sensor]} · {FSHORT[fault]}' for sensor in FINE_SENSORS for fault in FAULTS])
for row, sensor in enumerate(FINE_SENSORS, 1):
    for col, fault in enumerate(FAULTS, 1):
        profile = fine_profile[(fine_profile.sensor == sensor) & (fine_profile.fault_type == fault)].sort_values('bin_5min')
        x_hours = -(profile.bin_5min.to_numpy() + 0.5) * BIN_SEC / HOUR
        fig.add_scatter(x=np.r_[x_hours, x_hours[::-1]], y=np.r_[profile.q75, profile.q25[::-1]], fill='toself', mode='none',
                        fillcolor='rgba(120,120,120,.18)', showlegend=False, row=row, col=col)
        fig.add_scatter(x=x_hours, y=profile['median'], mode='lines', line=dict(color=[C_OK, C_FAIL][row - 1], width=2),
                        showlegend=False, hovertemplate='%{x:.2f}시간 전<br>중앙값 %{y:.2f}<extra></extra>', row=row, col=col)
        fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1), row=row, col=col)
fig.update_xaxes(title_text='고장까지 남은 시간(시간)', row=len(FINE_SENSORS))
fig.update_layout(title='그림 4-7a. 고장 직전 6시간 (4초 원본, 셔터 1, 5분 단위) — 사건 간 중앙값과 사분위 범위')
show(fig, 640)

# ── 사건별 직전 변화량 = (직전 10분 중앙값) − (2~6시간 전 중앙값) ────────
event_change_rows = []
for (event_id, fault_type), event_bins in fine_window.groupby(['event', 'fault_type']):
    change_row = {'event': event_id, 'fault_type': fault_type}
    for sensor in FINE_SENSORS:
        last_10min = event_bins.loc[event_bins.bin_5min < 2, sensor].median()                                  # 직전 10분
        early_2_6h = event_bins.loc[(event_bins.bin_5min >= 24) & (event_bins.bin_5min < 72), sensor].median()  # 2~6시간 전
        change_row[sensor + '_d'] = last_10min - early_2_6h
    event_change_rows.append(change_row)
event_changes = pd.DataFrame(event_change_rows)
change_table = []
for fault in FAULTS:
    fault_changes = event_changes[event_changes.fault_type == fault]
    for sensor in FINE_SENSORS:
        change_values = fault_changes[sensor + '_d'].dropna()
        change_table.append({'고장 종류': FSHORT[fault], '센서': NAMES[sensor], '전체 기록': int((events.fault_type == fault).sum()),
                             '직전 10분·2~6시간 전 모두 있음': len(change_values),
                             '변화량 중앙값(z)': round(change_values.median(), 2) if len(change_values) else np.nan,
                             '2 넘게 하락(%)': round((change_values < -2).mean() * 100, 1) if len(change_values) else np.nan,
                             '2 넘게 상승(%)': round((change_values > 2).mean() * 100, 1) if len(change_values) else np.nan})
display(pd.DataFrame(change_table))

고장 24시간 전 원본 중 셔터 1: 3,576,034행, 냉각 압력 z 계산됨: 96.8%


,고장 종류,센서,전체 기록,직전 10분·2~6시간 전 모두 있음,변화량 중앙값(z),2 넘게 하락(%),2 넘게 상승(%)
0,압력 저하,냉각 압력,741,216,-0.03,1.4,2.8
1,압력 저하,냉각 유량,741,216,-0.00,5.1,5.1
2,압력 과다,냉각 압력,333,41,0.11,2.4,9.8
3,압력 과다,냉각 유량,333,41,0.00,9.8,17.1
4,누설,냉각 압력,137,43,-0.02,9.3,0.0
5,누설,냉각 유량,137,43,-0.02,11.6,4.7


In [14]:
# 그림 4-7b: 고장 종류마다 직전 1시간 기록이 가장 많은 사건 3개의 4초 냉각 압력
fig = make_subplots(rows=1, cols=3, subplot_titles=ORDER, shared_yaxes=True)
example_colors = ['#2a78d6', '#eb6834', '#4c9a52']
for col, fault_type in enumerate(FAULTS, 1):
    candidates = []                                            # (직전 1시간 행 수, 장비, 고장 시각)
    for fault in events[events.fault_type == fault_type].itertuples():
        row_idx = prefault_rows_by_tool.get(fault.Tool)
        if row_idx is None:
            continue
        tool_time = prefault_etch.time.to_numpy()[row_idx]
        rows_last_hour = np.searchsorted(tool_time, fault.time) - np.searchsorted(tool_time, fault.time - HOUR)
        candidates.append((rows_last_hour, fault.Tool, fault.time))
    for rank, (rows_last_hour, tool, fault_time) in enumerate(sorted(candidates, reverse=True)[:3]):
        example = prefault_etch[(prefault_etch.Tool == tool) & (prefault_etch.time >= fault_time - 2 * HOUR)
                                & (prefault_etch.time < fault_time)]
        fig.add_scatter(x=(example.time - fault_time) / 60, y=example.FLOWCOOLPRESSURE_z, mode='lines',
                        line=dict(width=1, color=example_colors[rank]), name=f'{tool} · {FSHORT[fault_type]}', row=1, col=col)
fig.update_xaxes(title_text='고장까지 남은 시간(분)')
fig.update_yaxes(title_text='냉각 압력 z', col=1)
fig.update_layout(title='그림 4-7b. 사건 예시 — 직전 1시간 기록이 가장 많은 사건 3개의 냉각 압력 (4초, 셔터 1)',
                  legend=dict(orientation='h', y=-0.25))
show(fig, 480)

### 4-7c. 고장 전 몇 시간부터 달라지는가 — 시간 창별 구분력 (Q3)

4-6·4-7은 궤적 모양을 봤다. 여기서는 고장 전 시간을 **창**(0~1시간, 1~3시간, … 3~7일)으로 나눠, 창마다 셔터 1 구간의 z값이 평소와 얼마나 다른지 AUC로 잰다.

- **평소**: 앞뒤 고장 기록에서 모두 **7일 넘게** 떨어진 구간. 비교 창(최대 7일)과 겹치지 않게 3-2보다 엄격하게 잡았다.
- **고장 직후 제외**: 앞 고장에서 24시간 안의 구간은 창에서 뺀다. 고장 직후 상태가 섞이지 않게 하려는 것이다.
- 장비마다 AUC를 계산하고(창 구간·평소 구간 각 20개 이상), 장비별 값의 **중앙값**을 본다. 0.5는 차이 없음, 0.5보다 작으면 고장 전에 값이 낮다.
- 모델 학습 없이 비교만 한다.

In [15]:
WINDOWS_H = [(72, 168), (48, 72), (24, 48), (12, 24), (6, 12), (3, 6), (1, 3), (0, 1)]   # 먼 창 → 가까운 창
WINDOW_LABELS = [f'{lo_h}~{hi_h}시간' for lo_h, hi_h in WINDOWS_H]
WINDOW_SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE']

# 구간마다 직전 고장 이후 경과 시간, 다음 고장까지 남은 시간(초, 고장 종류 무관)
seconds_since_fault = np.full(len(df), np.inf)
seconds_until_fault = np.full(len(df), np.inf)
seg_start_all, seg_end_all = df.time_start.to_numpy(), df.time_end.to_numpy()
for tool, row_idx in rows_by_tool.items():
    fault_times = np.sort(events.loc[events.Tool == tool, 'time'].unique())
    if len(fault_times) == 0:
        continue
    prev_pos = np.searchsorted(fault_times, seg_start_all[row_idx])
    seconds_since_fault[row_idx] = np.where(prev_pos > 0, seg_start_all[row_idx] - fault_times[np.maximum(prev_pos - 1, 0)], np.inf)
    next_pos = np.searchsorted(fault_times, seg_end_all[row_idx])
    seconds_until_fault[row_idx] = np.where(next_pos < len(fault_times),
                                            fault_times[np.minimum(next_pos, len(fault_times) - 1)] - seg_end_all[row_idx], np.inf)
is_far_normal = np.minimum(seconds_since_fault, seconds_until_fault) > 7 * DAY   # 평소: 앞뒤 고장에서 7일 넘게
is_just_after_fault = seconds_since_fault <= DAY                                 # 고장 직후 24시간

window_rng = np.random.default_rng(0)
normal_idx_by_tool = {}
for tool, row_idx in rows_by_tool.items():
    normal_idx = row_idx[is_far_normal[row_idx] & is_shutter1[row_idx]]
    if len(normal_idx) > 50_000:                                                 # 평소 구간은 장비당 최대 5만 개 표본
        normal_idx = window_rng.choice(normal_idx, 50_000, replace=False)
    normal_idx_by_tool[tool] = normal_idx

window_rows = []
for fault in FAULTS:
    hours_to_fault = df[TTF[fault]].to_numpy() / HOUR
    for (lo_h, hi_h), window_label in zip(WINDOWS_H, WINDOW_LABELS):
        in_window = (hours_to_fault >= lo_h) & (hours_to_fault < hi_h) & ~is_just_after_fault & is_shutter1
        for sensor in WINDOW_SENSORS:
            tool_aucs = [auc(z_by_sensor[sensor][row_idx[in_window[row_idx]]], z_by_sensor[sensor][normal_idx_by_tool[tool]])
                         for tool, row_idx in rows_by_tool.items()]
            tool_aucs = [value for value in tool_aucs if np.isfinite(value)]
            window_rows.append({'고장': FSHORT[fault], '센서': NAMES[sensor], '창': window_label,
                                'AUC 중앙값': np.median(tool_aucs) if tool_aucs else np.nan, '장비 수': len(tool_aucs)})
window_auc = pd.DataFrame(window_rows)

for sensor in WINDOW_SENSORS:
    table = (window_auc[window_auc.센서 == NAMES[sensor]]
             .pivot(index='창', columns='고장', values=['AUC 중앙값', '장비 수']).reindex(WINDOW_LABELS))
    table = table.reindex(columns=pd.MultiIndex.from_product([['AUC 중앙값', '장비 수'], ORDER]))
    display(Markdown(f'**{NAMES[sensor]}** — 장비별 AUC 중앙값과 계산한 장비 수'))
    display(table.round(3))

# 그림 4-7c: 창별 AUC 중앙값 (가로축 왼쪽 = 먼 과거, 오른쪽 = 고장 직전)
fig = make_subplots(rows=1, cols=2, subplot_titles=[NAMES[sensor] for sensor in WINDOW_SENSORS], shared_yaxes=True)
for col, sensor in enumerate(WINDOW_SENSORS, 1):
    for fault_name in ORDER:
        line_data = window_auc[(window_auc.센서 == NAMES[sensor]) & (window_auc.고장 == fault_name)].set_index('창').reindex(WINDOW_LABELS)
        fig.add_trace(go.Scatter(x=WINDOW_LABELS, y=line_data['AUC 중앙값'], mode='lines+markers', name=fault_name,
                                 legendgroup=fault_name, showlegend=col == 1, line=dict(color=FCOLOR[fault_name], width=2),
                                 customdata=line_data['장비 수'],
                                 hovertemplate='%{x}<br>AUC %{y:.3f}<br>장비 %{customdata}대<extra>' + fault_name + '</extra>'),
                      row=1, col=col)
    fig.add_hline(y=.5, line=dict(color='#999', dash='dot', width=1), row=1, col=col)
fig.update_xaxes(title_text='고장 전 시간 창 (오른쪽이 고장 직전)')
fig.update_yaxes(title_text='AUC 중앙값 (0.5 = 차이 없음)', row=1, col=1)
fig.update_layout(title='그림 4-7c. 고장 전 시간 창별 구분력 (셔터 1, 장비별 AUC 중앙값)', legend=dict(orientation='h', y=-0.25))
show(fig, 480)

**냉각 압력** — 장비별 AUC 중앙값과 계산한 장비 수

AUC 중앙값                장비 수            
           압력 저하  압력 과다     누설 압력 저하 압력 과다    누설
창                                               
72~168시간   0.435  0.495  0.334  19.0  18.0  14.0
48~72시간    0.449  0.546  0.358  19.0  16.0  13.0
24~48시간    0.411  0.492  0.388  19.0  16.0  13.0
12~24시간    0.373  0.519  0.450  19.0  14.0  11.0
6~12시간     0.391  0.522  0.405  17.0  13.0   7.0
3~6시간      0.460  0.575  0.440  16.0   6.0   4.0
1~3시간      0.408  0.570  0.533  12.0   7.0   4.0
0~1시간      0.388  0.610  0.561   9.0   2.0   2.0

**냉각 유량** — 장비별 AUC 중앙값과 계산한 장비 수

AUC 중앙값                장비 수            
           압력 저하  압력 과다     누설 압력 저하 압력 과다    누설
창                                               
72~168시간   0.500  0.493  0.499  19.0  18.0  14.0
48~72시간    0.494  0.451  0.518  19.0  16.0  13.0
24~48시간    0.479  0.454  0.480  19.0  16.0  13.0
12~24시간    0.466  0.438  0.503  19.0  14.0  11.0
6~12시간     0.468  0.466  0.543  17.0  13.0   7.0
3~6시간      0.473  0.406  0.494  16.0   6.0   4.0
1~3시간      0.476  0.404  0.473  12.0   7.0   4.0
0~1시간      0.470  0.161  0.453   9.0   2.0   2.0

**해석**: 고장 종류마다 달라지는 시점이 다르다.

| 고장 | 달라지는 센서 | 언제부터 | 모양 |
|---|---|---|---|
| 압력 저하 | 냉각 압력 (낮음) | **일주일 전부터 이미** (3~7일 전 0.435) | 모든 창에서 0.37~0.46으로 비슷하다. 가장 큰 곳은 12~24시간 전(0.373). 직전에 더 떨어지지 않는다 → **수준 차이** |
| 압력 과다 | 냉각 유량 (낮음) | **약 3~6시간 전부터** (0.406) | 12시간 전까지 0.44~0.47이다가 직전으로 갈수록 떨어진다. 0~1시간 0.161(장비 2대뿐) → **서서히 다가오는 변화** |
| 누설 | 냉각 압력 (낮음) | 일주일 전에 가장 큼 (0.334) | 고장에 가까울수록 오히려 차이가 줄어든다. 누설 전 시기의 압력 수준을 반영한 것으로 보인다 |

- 고장 직전 창은 계산할 수 있는 장비가 적다(0~1시간 2~9대). 4-8에서 보듯 고장 직전에 기록이 없는 경우가 많기 때문이다. 6시간 넘는 창부터 장비 7대 이상이 들어간다.
- 냉각 압력의 압력 과다 AUC는 0.49~0.61로 대부분 0.5 근처다. 압력 과다는 압력보다 **유량**을 봐야 한다.
- **라벨 창은 고장 종류별로 다르게** 잡는 것이 맞다. 압력 과다는 6시간 정도, 압력 저하·누설은 짧은 창보다 "최근 대비 변화"로 보는 쪽이 낫다.

### 4-8. 고장 직전에 기록이 있는가 (Q4)

고장 시각은 장비를 세운 시점이라 직전에 이미 멈춰 있으면 기록이 없다. 고장 기록마다 직전 10분·1시간·24시간 중 **셔터 1 구간이 덮는 시간의 비율**을 구한다. 모든 셔터 기록 기준 값은 2-3에 있다. 0%면 그 사건은 원리적으로 미리 볼 수 없다.

In [16]:
# 고장 기록마다 직전 10분·1시간·24시간이 스텝 구간 기록으로 얼마나 덮여 있는지
seg_end_all = df.time_end.to_numpy()
coverage_rows = []
for tool, row_idx in rows_by_tool.items():
    tool_start, tool_end = seg_start_all[row_idx], seg_end_all[row_idx]
    longest_segment = (tool_end - tool_start).max()            # 창보다 먼저 시작해 창에 걸치는 구간까지 찾기 위한 여유
    for fault in events[events.Tool == tool].itertuples():
        coverage_row = {'fault_type': FSHORT[fault.fault_type]}
        for window_sec, window_name in [(600, '10분'), (HOUR, '1시간'), (DAY, '24시간')]:
            win_lo = np.searchsorted(tool_start, fault.time - window_sec - longest_segment, 'left')
            win_hi = np.searchsorted(tool_start, fault.time, 'left')
            # 구간과 창이 겹치는 시간의 합
            overlap = np.clip(np.minimum(tool_end[win_lo:win_hi], fault.time)
                              - np.maximum(tool_start[win_lo:win_hi], fault.time - window_sec), 0, None).sum()
            coverage_row[window_name] = min(overlap / window_sec, 1.0)   # 창 중 기록이 있는 비율 (최대 1)
        coverage_rows.append(coverage_row)
coverage = pd.DataFrame(coverage_rows)
coverage_table = coverage.groupby('fault_type').agg(
    기록=('10분', 'size'),
    **{f'직전 {window_name} 기록 없음(%)': (window_name, lambda covered: round((covered == 0).mean() * 100, 1))
       for window_name in ['10분', '1시간', '24시간']},
    **{f'직전 {window_name} 덮인 비율 중앙값(%)': (window_name, lambda covered: round(covered.median() * 100, 1))
       for window_name in ['1시간', '24시간']}).reindex(ORDER)
display(coverage_table)

,기록,직전 10분 기록 없음(%),직전 1시간 기록 없음(%),직전 24시간 기록 없음(%),직전 1시간 덮인 비율 중앙값(%),직전 24시간 덮인 비율 중앙값(%)
fault_type,,,,,,
압력 저하,741,63.4,37.0,1.1,19.2,24.1
압력 과다,333,83.2,70.6,31.5,0.0,11.4
누설,137,64.2,43.8,6.6,15.3,22.5


### 4-9. 고장 간격 분포 (Q4)

같은 장비에서 **같은 종류 기록 사이 간격**과 **종류와 무관하게 다음 고장까지 간격**을 본다. 간격이 라벨 창(24시간)보다 짧으면 고장 전 라벨이 앞 고장의 정비·재가동 구간과 겹친다. 같은 고장이 짧은 간격으로 반복 기록되는 것도 여기서 보인다.

In [17]:
# 고장 기록 사이 간격(시간): 같은 종류끼리 / 종류 무관
events_by_type = events.sort_values(['Tool', 'fault_type', 'time']).copy()
events_by_type['gap_h'] = events_by_type.groupby(['Tool', 'fault_type']).time.diff() / HOUR
events_any_type = events.sort_values(['Tool', 'time']).copy()
events_any_type['gap_h'] = events_any_type.groupby('Tool').time.diff() / HOUR

def gap_row(row_name, gap_hours):
    """간격 분포 요약 한 줄 (분위수, 1시간·24시간 미만 비율)."""
    gap_hours = gap_hours.dropna()
    return {'구분': row_name, '간격 수': len(gap_hours), '10%(시간)': round(gap_hours.quantile(.1), 1),
            '중앙값(시간)': round(gap_hours.median(), 1), '90%(일)': round(gap_hours.quantile(.9) / 24, 1),
            '최대(일)': round(gap_hours.max() / 24, 1), '1시간 미만(%)': round((gap_hours < 1).mean() * 100, 1),
            '24시간 미만(%)': round((gap_hours < 24).mean() * 100, 1)}

gap_table = pd.DataFrame([gap_row(f'같은 종류 · {FSHORT[fault]}', events_by_type.loc[events_by_type.fault_type == fault, 'gap_h'])
                          for fault in FAULTS]
                         + [gap_row('종류 무관 · 다음 고장', events_any_type.gap_h)]).set_index('구분')
display(gap_table)

# 그림 4-9: 같은 종류 간격 분포 (로그 눈금)
TICKS = [0.1, 1, 6, 24, 24 * 7, 24 * 30, 24 * 180]
fig = go.Figure()
for fault in FAULTS:
    fault_gaps = events_by_type.loc[(events_by_type.fault_type == fault) & (events_by_type.gap_h > 0), 'gap_h']
    fig.add_histogram(x=np.log10(fault_gaps), name=FSHORT[fault], marker_color=FCOLOR[FSHORT[fault]], opacity=.6,
                      xbins=dict(size=0.2))
for line_hours, line_label in [(1, '1시간'), (24, '24시간')]:
    fig.add_vline(x=np.log10(line_hours), line=dict(color='#444', dash='dash'), annotation_text=line_label)
fig.update_xaxes(tickvals=np.log10(TICKS), ticktext=['6분', '1시간', '6시간', '1일', '1주', '1달', '6달'],
                 title_text='같은 종류 앞 기록과의 간격 (로그 눈금)')
fig.update_layout(title='그림 4-9. 같은 종류 고장 기록 사이 간격', barmode='overlay', yaxis_title='간격 수')
show(fig, 440)

# 묶는 간격에 따라 사건 수가 어떻게 바뀌는지: 앞 기록과 간격이 기준보다 크면 새 사건
episode_rows = []
for group_gap_sec, gap_label in [(0, '묶지 않음'), (HOUR, '1시간'), (6 * HOUR, '6시간'), (DAY, '1일')]:
    gap_seconds = events_by_type.groupby(['Tool', 'fault_type']).time.diff()
    is_new_episode = gap_seconds.isna() | (gap_seconds > group_gap_sec)
    episode_row = {'묶는 간격': gap_label,
                   **{FSHORT[fault]: int(is_new_episode[events_by_type.fault_type == fault].sum()) for fault in FAULTS}}
    episode_row['합계'] = int(is_new_episode.sum())
    episode_rows.append(episode_row)
display(pd.DataFrame(episode_rows).set_index('묶는 간격'))

,간격 수,10%(시간),중앙값(시간),90%(일),최대(일),1시간 미만(%),24시간 미만(%)
구분,,,,,,,
같은 종류 · 압력 저하,722,2.4,24.5,10.8,310.3,5.5,49.6
같은 종류 · 압력 과다,314,0.5,55.9,31.9,360.2,17.5,39.8
같은 종류 · 누설,123,0.4,6.2,54.0,361.3,20.3,65.0
종류 무관 · 다음 고장,1191,0.5,19.7,13.3,308.3,14.8,53.9


,압력 저하,압력 과다,누설,합계
묶는 간격,,,,
묶지 않음,741,333,137,1211
1시간,701,278,112,1091
6시간,582,245,77,904
1일,383,208,57,648


## 5. 인사이트 → 모델링 반영

### 5-0. 분석 목적 질문에 대한 답

| 질문 | 답 | 근거 | 정한 것 |
|---|---|---|---|
| Q1. 레시피·스텝은 센서 값에 얼마나 영향을 주는가? | **매우 크다.** 센서 흔들림의 대부분이 공정 단계에서 나온다 | 셔터 1 기준 설명 비율 중앙값: 스텝만 0.14, 레시피+스텝 0.735, 장비·레시피·스텝·셔터 0.87. 장비 자체는 0.045 (4-2) | 센서 값을 그대로 쓰지 않고 **같은 장비·레시피·스텝·셔터의 평소 대비 z값**을 쓴다 |
| Q2. 어떤 센서가 고장 전과 평소를 구분하는가? | **냉각 압력**만 뚜렷하다. 고장 전에 평소보다 낮다 | 셔터 1 기준 \|AUC−0.5\| 최대 0.209. 나머지 공정 센서는 대부분 0.45~0.55. 카운터는 높아 보이지만 정비 때 초기화되는 누적값이라 노화 효과인지 확인이 필요 (4-5, 3-3) | 냉각 압력 중심. 압력 과다는 냉각 유량을 함께 본다. 회전은 빼고, 카운터는 넣은 경우·뺀 경우를 비교한다 |
| Q3. 징후는 몇 시간 전부터 보이는가? 갑자기인가, 서서히인가? | **고장 종류마다 다르다.** 갑작스러운 하락은 거의 없다 | 압력 저하·누설: 냉각 압력이 일주일 전부터 이미 낮고 직전에 더 변하지 않음. 압력 과다: 냉각 유량이 약 3~6시간 전부터 서서히 떨어짐 (4-6, 4-7, 4-7c) | 라벨 창을 고장 종류별로 정한다(압력 과다 약 6시간). 입력은 현재 수준보다 **최근 대비 변화**를 시험한다 |
| Q4. 고장 직전에 기록이 있는가? 고장 간격은? | **직전 기록이 없는 고장이 많고, 같은 고장이 반복 기록된다** | 직전 1시간 식각 기록 없음 37~71%(압력 과다 70.6%). 24시간 안 재기록이 많아 1일로 묶으면 1,211건 → 648건 (4-8, 4-9) | **하루 묶음 사건**으로 평가한다. 기록 없는 사건은 따로 표시한다 |
| Q5. 장비·M01/M02·셔터에 따라 다른가? | **모두 다르다** | 고장 상위 3대가 51.5%. M01–M02 공유 레시피 거의 0. 식각 중과 셔터 0의 센서 차이 표준편차 1.4~2.0배. 장비 4대를 자세히 보면 고장 전 압력 방향이 장비마다 반대이고, 압력 수준이 몇 달 단위로 이동한다(부록 `eda_tools.ipynb`) (4-1, 4-3, 2-3) | **셔터 1만** 쓴다. **장비 단위로 검증**한다. M01/M02별 또는 장비별 기준을 쓴다 |

### 5-1. 실제 결과 요약

| 질문 | 결과 | 모델링 반영 |
|---|---|---|
| 품질. 중복·이상치 | 원본 완전 중복 0.007%(제거), 같은 시각·다른 값 0.14%(남김), 고장 기록 중복 25건(제거). 셔터 1에서 냉각 압력 > 5 구간(48개)의 27.1%가 고장 24시간 전(평균 7.3%). 회전 속도 IQR×3 밖 11.4%, 고장과 무관 (3-3) | 지울 이상치 없음. 냉각 압력 극단값은 신호로 남기고, 회전은 입력에서 뺀다 |
| Q1. 스텝 영향 | 셔터 1 기준 센서 분산 중 스텝이 설명하는 비율 중앙값 **0.14**, 레시피+스텝 **0.735**, 장비·레시피·스텝·셔터 **0.87**. 장비 자체는 0.045. 셔터 효과가 빠져 스텝 단독 값은 작지만 레시피와 함께 보면 여전히 대부분을 설명한다 (1분 요약·모든 셔터에서는 스텝 0.44, 레시피+스텝 0.64) | 센서 값을 그대로 쓰지 않고 **같은 레시피·스텝·셔터의 평소 값 대비 z값**으로 쓴다 |
| Q2. 구분력 | 셔터 1 기준 장비별 AUC 중앙값: **냉각 압력** 압력 저하 0.365, 누설 0.291(고장 전에 낮음). 전체 구간보다 셔터 1에서 더 잘 구분(0.154 → 0.209). 나머지 센서는 대부분 0.45~0.55 | 냉각 압력을 핵심 후보로. **셔터 1 구간만** 쓰는 쪽이 유리 |
| Q2. 카운터 | 실드 사용량1·2가 0.148·0.093으로 높다 | 노화 효과일 수 있어 모델에 넣은 경우·뺀 경우를 비교 |
| Q3. 시점·모양 | 4초 원본에서 직전 10분과 2~6시간 전을 비교해도 냉각 압력 변화량 중앙값은 거의 0(압력 저하 −0.03). z가 2 넘게 떨어진 사건은 압력 저하의 1.4%뿐. 48시간 궤적에서도 직전 3시간에 뚜렷한 하락이 없다 | 선행 연구가 말한 "갑작스러운 하락"은 셔터 1 기록에서 거의 보이지 않는다. 냉각 압력의 차이는 **24시간보다 이전부터 있는 수준 차이**에 가깝다. 라벨 창을 줄이는 것만으로 좋아진다고 장담할 수 없어 1·6·24·48시간을 비교한다 |
| Q3. 시간 창 | 압력 저하: 냉각 압력 AUC가 3~7일 전 0.435, 12~24시간 전 0.373으로 창과 상관없이 낮음. 압력 과다: 냉각 유량이 12시간 전까지 0.44~0.47, 3~6시간 전 0.406, 0~1시간 0.161(장비 2대). 누설: 3~7일 전 0.334로 가장 큼 (4-7c) | 압력 과다만 "다가올수록 커지는" 신호. 라벨 창을 고장 종류별로 다르게 잡는다 |
| Q4. 직전 기록 | 직전 1시간 식각 기록 없음: 압력 저하 37.0%, **압력 과다 70.6%**, 누설 43.8%(모든 기록 기준 31.6 / 59.8 / 29.9%). 직전 10분은 압력 저하도 63.4%가 비어 있다 | 기록 없는 사건은 원리적으로 예측 불가 → 평가에서 따로 표시하거나 뺀다 |
| Q4. 고장 간격 | 같은 종류 앞 기록과 24시간 미만: 압력 저하 49.6%, 누설 65.0%. 1일 간격으로 묶으면 1,211건 → 648건 | 사건 정의(묶는 간격)를 먼저 정한다. 24시간 라벨은 앞 고장 직후 구간과 겹친다 |
| Q4. 데이터 끝 | 기록 끝 24시간 안에서 TTF가 모두 비어 판단 불가한 구간은 원본 행의 0.2% | 라벨에서 뺀다(영향은 작음) |
| Q5. 장비 | 고장 상위 3대(M02_03, M01_09, M01_08)가 51.5%. M02가 57.7% | 장비 단위 검증 유지 |
| Q5. M01/M02 | M01끼리·M02끼리 공유 레시피 평균 13.6개, M01–M02 사이 0.1개, 같은 번호끼리도 0.1개 | 두 종류의 장비(또는 공정 계열)로 본다. 따로 학습하거나 구분 열을 넣어 비교 |
| Q5. 장비별 패턴 (부록) | `eda_tools.ipynb`: 압력 저하 전 냉각 압력이 16대 중 11대는 낮고 5대는 높음(M01은 대부분 낮고 M02는 대부분 높음). 고장이 거의 없는 장비도 냉각 압력 수준이 몇 달 단위로 이동 | 처음 보는 장비에 하나의 방향을 기대하기 어렵다. 장비별 기준, 최근 대비 변화 입력 |
| Q5. 셔터 | 식각 중(셔터 1)과 셔터 0의 센서 값 차이가 표준편차 1.4~2.0배(냉각 압력 1.50)로 사실상 다른 상태. 원본 행 기준 셔터 1이 61.0%. 압력 과다 24시간 전은 48.6%로 낮고 셔터 0이 47.3% | 셔터 1만 쓴다(2-3). 비교는 공정해지지만 "멈춰 있는 시간이 늘었다"는 힌트는 사라진다 → 셔터 0 시간 비율을 별도 입력으로 비교 |

**한 줄 요약**: 스텝·레시피가 센서 값의 대부분을 설명하므로 스텝별 z값이 필수다. 그렇게 보면 냉각 압력이 고장 전 24시간에 평소보다 낮다는 신호가 있다. 다만 고장 직전 급변은 거의 보이지 않고, 고장의 37~71%는 직전 1시간 식각 기록이 없다.

### 5-2. 컬럼 후보 결정표

셔터 1 기준 AUC에서 고장 3종 중 **0.5에서 가장 멀리 떨어진 값**으로 줄 세운 **초안**이다. 학습 성능으로 정한 최종 목록이 아니다. 기준: 0.10 이상 우선 후보, 0.05~0.10 조건부, 0.05 미만 보류.

In [18]:
# 센서별 구분력 = 고장 3종 중 0.5에서 가장 먼 AUC (셔터 1, 장비별 중앙값 기준)
deviation_shutter1 = (auc_median_shutter1 - .5).abs()
best_fault_pos = deviation_shutter1.to_numpy().argmax(axis=1)                  # 센서마다 가장 잘 구분하는 고장의 위치
decision_table = pd.DataFrame({
    '센서': [NAMES[sensor] for sensor in SENSORS], '컬럼': SENSORS,
    '최대 |AUC−0.5|': deviation_shutter1.max(axis=1).round(3).to_numpy(),
    '가장 잘 구분하는 고장': deviation_shutter1.idxmax(axis=1).to_numpy(),
    '방향': np.where(auc_median_shutter1.to_numpy()[np.arange(len(SENSORS)), best_fault_pos] < .5, '고장 전 낮음', '고장 전 높음')})
# 판정 기준: 0.10 이상 우선 후보, 0.05~0.10 조건부, 0.05 미만 보류
decision_table['판정'] = pd.cut(decision_table['최대 |AUC−0.5|'], [-1, .05, .10, 1], labels=['보류', '조건부', '우선 후보'])
decision_table['비고'] = np.where(decision_table.컬럼.isin(COUNTERS),
                                 '누적 카운터: 장비 노화 효과일 수 있음, 정비 때 초기화 여부 확인 필요', '')
display(decision_table.sort_values('최대 |AUC−0.5|', ascending=False).reset_index(drop=True))

,센서,컬럼,최대 |AUC−0.5|,가장 잘 구분하는 고장,방향,판정,비고
0,냉각 압력,FLOWCOOLPRESSURE,0.209,누설,고장 전 낮음,우선 후보,
1,실드 사용량1,ETCHAUXSOURCETIMER,0.148,누설,고장 전 낮음,우선 후보,"누적 카운터: 장비 노화 효과일 수 있음, 정비 때 초기화 여부 확인 필요"
2,실드 사용량2,ETCHAUX2SOURCETIMER,0.093,압력 저하,고장 전 높음,조건부,"누적 카운터: 장비 노화 효과일 수 있음, 정비 때 초기화 여부 확인 필요"
3,억제 전류,ETCHSUPPRESSORCURRENT,0.071,압력 저하,고장 전 높음,조건부,
4,냉각 유량,FLOWCOOLFLOWRATE,0.045,압력 과다,고장 전 낮음,보류,
5,아르곤 유량(소스),ETCHGASCHANNEL1READBACK,0.043,압력 저하,고장 전 높음,보류,
6,챔버 압력,IONGAUGEPRESSURE,0.038,누설,고장 전 높음,보류,
7,빔 전압,ETCHBEAMVOLTAGE,0.032,누설,고장 전 높음,보류,
8,아르곤 유량(PBN),ETCHPBNGASREADBACK,0.027,누설,고장 전 낮음,보류,
9,회전 각도,ACTUALROTATIONANGLE,0.027,누설,고장 전 낮음,보류,


### 5-3. 적용 범위·한계·다음 단계

- AUC는 **같은 장비 안에서** 고장 전과 평소를 비교한 값이다. 처음 보는 장비로 일반화되는지는 모델 검증(`models/`)에서 따로 확인해야 한다.
- 구간 단위 분석이라 긴 구간과 짧은 구간이 같은 무게다(궤적은 원본 행 수로 가중). 셔터 1 원본 행만 먼저 거른 뒤 구간을 만들었으므로 구간 안에 다른 셔터 값이 섞이지 않는다(2-3).
- 센서는 표준화돼 있어 **실제 단위의 관리 기준은 만들 수 없다.** 평소 대비 이탈로 경보 기준을 정한다.
- 고장 시각은 정비 정지 시각이다. 실제 이상 시작은 더 이를 수 있고 제공되지 않는다.
- '평소'는 고장에서 24시간 넘게 떨어진 구간일 뿐 정상으로 확정된 라벨이 아니다. 평소 기준은 장비 전체 기간에서 구했으므로, 모델에서는 **학습 장비·학습 기간에서만** 다시 계산해야 정보가 새지 않는다.
- 셔터 1에서 조건(장비·레시피·스텝·셔터) 조합 3,110개 중 평소 구간이 30개 이상인 조합은 598개다. 드문 조합은 z값이 비어 있다(냉각 압력 기준 약 2~3%).
- M01·M02의 뜻과 셔터 값의 뜻은 공식 설명에 없다. 여기 적은 해석은 데이터와 선행 연구에서 나온 추정이다.
- 다음 단계: (1) 셔터 1·스텝별 z값·사건 정의(1일 묶음) 적용, 셔터 0 시간 비율 입력 비교, (2) 라벨 창 1·6·24·48시간 비교, (3) 직전 기록 없는 사건을 뺀 평가, (4) 카운터 포함·제외 비교, (5) M01/M02 분리 비교. 시험 장비 5대는 정답이 없어 쓰지 않는다.

### 재현 경로

1. 원본을 `data/raw/`에 두고 압축을 푼다(루트 `README.md` 참고).
2. `python -X utf8 src/make_processed.py`: `step/`, `prefault/`, `faults.csv`를 만든다.
3. 이 노트북을 처음부터 실행한다. 그림·표는 파일로 저장하지 않고 노트북 출력에만 남는다.